# GEO V7: исправленная версия, финальное обучение не более 45 минут

**45 минут относятся к обучению финального решения, а не ко всему Run All.**
Предварительное сравнение MLP/CNN и предобученных моделей идёт отдельно.
Полный первый запуск может занять больше 45 минут. Не выдаём его время
за время одного финального обучения.

Новый финальный этап каждый раз обучается заново. Его таймер включает
запуск процессов, загрузку моделей, обучение, сохранение FP32 EMA и
завершение процессов. До 45 минут есть запас: обучение ограничено
меньшим внутренним бюджетом, зависшие процессы останавливает родитель.
Нет фиктивного нулевого времени из-за готовых result.json.

**Данные:** предварительные модели — только train, оценка на всём val.
Финальные модели — train + val (val повторяется дважды), как разрешено
для командного этапа. Это не индивидуальный checkpoint.

**Test используется только в последней ячейке получения предсказаний.**
Не анализируем его дефекты/распределение, не строим псевдометки,
не подбираем по нему веса, аугментации или BatchNorm. Полностью исключить
test и одновременно создать submission невозможно: для CSV нужны изображения.

Не импортируются готовые модели или прогнозы других команд. Архитектуры
и аугментации — разрешённые torchvision; веса только ImageNet.


## Запуск

1. Kaggle Input: **официальный полный датасет**, старые архивы не требуются.
2. **T4 ×2, Internet ON**, затем Save Version → Run All.
3. Для защиты на уже подготовленной сессии можно повторить отдельно
   ячейку **«Финальное обучение и официальный таймер»**, затем предсказания.
   Финальное обучение при этом действительно запускается заново.

Предварительные эксперименты не включаются в лимит финального обучения,
но остаются в коде/отчёте. Срок из присланных правил — **10 октября 2026,
14:00 МСК**; заранее оставь время на CSV и Moodle.


In [ ]:
import os, time, hashlib, importlib, sys, traceback
from pathlib import Path
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
NOTEBOOK_STARTED = time.time()

DATA_DIR_OVERRIDE = None
RUN_DIR = Path("/kaggle/working/geo_v7_45")
CODE_DIR = Path("/kaggle/working/geo_v7_45_code")
CACHE_DIR = Path("/tmp/geo_v7_45_images")
BASELINE_EPOCHS = 8

import torch
if not torch.cuda.is_available():
    raise RuntimeError("Включи GPU в Kaggle. CPU не подходит для защиты с этим лимитом.")
print("GPU:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
if torch.cuda.device_count() < 2:
    print("Только один GPU: времени меньше, финальное обучение может не завершиться.")


In [ ]:
# Только собственный исполняемый код. Изображений, CSV и весов во вложении нет.
EMBEDDED_SOURCES = {'geo_worker.py': '"""Isolated, reproducible PyTorch trials. Run ``python geo_worker.py --job job.json``.\n\nScreening uses only a deterministic split of train.csv. Its four-view score is\na synthetic robustness proxy, never an estimate measured on official val.csv.\nIncomplete runs are deliberately non-resumable; use a new output directory.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport json\nimport math\nimport os\nimport random\nimport re\nimport time\nfrom pathlib import Path\n\nos.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")\n\nimport numpy as np\nimport pandas as pd\nfrom PIL import Image\nimport torch\nfrom torch import nn\nfrom torch.utils.data import DataLoader, Dataset\nimport torchvision\nfrom torchvision import models, transforms as T\nfrom torchvision.transforms import functional as TF\n\n\nMEAN = (0.485, 0.456, 0.406)\nSTD = (0.229, 0.224, 0.225)\nINNER_SPLIT_SEED = 20261003\nMODEL_NAMES = {"mlp", "cnn_plain", "cnn_bn", "cnn_gap", "resnet18", "resnet34", "resnet50"}\n\n\ndef seed_everything(seed):\n    """Seed this process; the scheduler must isolate its visible CUDA device."""\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n    torch.set_num_threads(2)\n    try:\n        torch.set_num_interop_threads(2)\n    except RuntimeError:\n        pass  # May already be configured in an importing process.\n    torch.backends.cudnn.benchmark = False\n    torch.backends.cudnn.deterministic = True\n    torch.use_deterministic_algorithms(True, warn_only=True)\n\n\ndef _worker_seed(_worker_id):\n    seed = torch.initial_seed() % (2**32)\n    np.random.seed(seed)\n    random.seed(seed)\n    torch.set_num_threads(1)\n\n\ndef _sha256(path):\n    digest = hashlib.sha256()\n    with Path(path).open("rb") as stream:\n        for block in iter(lambda: stream.read(1024 * 1024), b""):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef _json_write(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_name(path.name + ".tmp")\n    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")\n    os.replace(temporary, path)\n\n\ndef _read_csv(path, columns):\n    frame = pd.read_csv(path, dtype=str, keep_default_na=False)\n    if list(frame.columns) != list(columns):\n        raise ValueError(f"{Path(path).name}: expected columns {list(columns)}, got {list(frame.columns)}")\n    if frame.empty:\n        raise ValueError(f"{Path(path).name}: table is empty")\n    if "id" in frame:\n        ids = frame["id"]\n        invalid = ids.map(lambda x: not x or x in {".", ".."} or any(c in x for c in "/\\\\\\x00:") or Path(x).name != x)\n        if invalid.any() or ids.duplicated().any():\n            raise ValueError(f"{Path(path).name}: IDs must be unique, literal filenames without paths")\n    if "label" in frame:\n        if not frame["label"].map(lambda x: bool(re.fullmatch(r"0|[1-9][0-9]*", x))).all():\n            raise ValueError(f"{Path(path).name}: labels must be nonnegative integers")\n        frame["label"] = frame["label"].astype(np.int64)\n    return frame\n\n\ndef load_tables(data_dir, phase="final"):\n    """Read validated metadata; ``phase=\'screen\'`` never opens val/test metadata."""\n    root = Path(data_dir)\n    classes_frame = _read_csv(root / "classes.csv", ["label", "class_name"])\n    classes_frame = classes_frame.sort_values("label").reset_index(drop=True)\n    if classes_frame["label"].tolist() != list(range(len(classes_frame))):\n        raise ValueError("classes.csv: class labels must be unique and contiguous from zero")\n    if classes_frame["class_name"].duplicated().any() or (classes_frame["class_name"].str.strip() == "").any():\n        raise ValueError("classes.csv: class names must be unique and nonempty")\n    result = {"classes": classes_frame["class_name"].tolist()}\n    names = ["train"] if phase == "screen" else ["train", "val", "test"]\n    signatures = {"classes.csv": _sha256(root / "classes.csv")}\n    split_ids, overlaps = {}, {}\n    for split in names:\n        columns = ["id"] if split == "test" else ["id", "label"]\n        frame = _read_csv(root / f"{split}.csv", columns)\n        if "label" in frame and not frame["label"].between(0, len(classes_frame) - 1).all():\n            raise ValueError(f"{split}.csv: label outside classes.csv range")\n        ids = set(frame["id"])\n        # Filenames are split-local. Repeated names do not establish image leakage.\n        for earlier, earlier_ids in split_ids.items():\n            overlaps[f"{earlier}:{split}"] = len(earlier_ids.intersection(ids))\n        split_ids[split] = ids\n        result[split] = frame\n        signatures[f"{split}.csv"] = _sha256(root / f"{split}.csv")\n    sample_path = root / "sample_submission_example.csv"\n    if phase != "screen" and sample_path.exists():\n        sample = pd.read_csv(sample_path, dtype=str, keep_default_na=False)\n        if list(sample.columns) != ["id", "label"]:\n            raise ValueError("Sample submission must have columns id,label")\n        result["sample_submission"] = sample\n        result["sample_order_matches_test"] = sample["id"].tolist() == result["test"]["id"].tolist()\n        signatures[sample_path.name] = _sha256(sample_path)\n    result["data_signature"] = signatures\n    result["split_id_overlaps"] = overlaps\n    return result\n\n\ndef _size(cfg):\n    return 32 if cfg["model"] == "mlp" else int(cfg.get("size", 96))\n\n\ndef build_transform(cfg, training=False, corruption=None):\n    """Only torchvision transforms; evaluation corruptions have fixed parameters."""\n    size = _size(cfg)\n    steps = [T.Resize((size, size), antialias=True)]\n    if training:\n        augment = cfg.get("augment", "standard")\n        if augment not in {"none", "basic", "standard", "robust"}:\n            raise ValueError(f"Unknown augmentation: {augment}")\n        if augment != "none":\n            steps += [T.RandomHorizontalFlip(), T.RandomVerticalFlip(),\n                      T.RandomChoice([T.RandomRotation((angle, angle)) for angle in (0, 90, 180, 270)]),\n                      T.ColorJitter(brightness=0.12, contrast=0.12, saturation=0.12, hue=0.02)]\n        if augment == "robust":\n            steps += [T.RandomApply([T.GaussianBlur(5, sigma=(0.2, 1.5))], p=0.25),\n                      T.RandomInvert(p=0.10),\n                      T.RandomApply([T.Resize((size // 2, size // 2), antialias=True),\n                                     T.Resize((size, size), antialias=True)], p=0.25)]\n    elif corruption == "blur":\n        steps += [T.GaussianBlur(5, sigma=(1.2, 1.2))]\n    elif corruption == "lowres":\n        steps += [T.Resize((size // 2, size // 2), antialias=True), T.Resize((size, size), antialias=True)]\n    elif corruption == "invert":\n        steps += [T.RandomInvert(p=1.0)]\n    elif corruption not in {None, "clean", "none"}:\n        raise ValueError(f"Unknown evaluation corruption: {corruption}")\n    return T.Compose(steps + [T.ToTensor(), T.Normalize(MEAN, STD)])\n\n\nclass _ScreenViews:\n    def __init__(self, cfg):\n        self.transforms = [build_transform(cfg, corruption=view) for view in ("clean", "blur", "lowres", "invert")]\n\n    def __call__(self, image):\n        return torch.stack([transform(image) for transform in self.transforms])\n\n\nclass _PredictionViews:\n    def __init__(self, cfg, tta):\n        if tta not in {"none", "d4", "fivecrop"}:\n            raise ValueError("tta must be none, d4, or fivecrop")\n        self.tta, self.size = tta, _size(cfg)\n        self.normal = build_transform(cfg)\n        self.tensor = T.Compose([T.ToTensor(), T.Normalize(MEAN, STD)])\n\n    def __call__(self, image):\n        if self.tta == "fivecrop":\n            expanded = TF.resize(image, [math.ceil(self.size * 1.14)] * 2, antialias=True)\n            return torch.stack([self.tensor(crop) for crop in TF.five_crop(expanded, self.size)])\n        tensor = self.normal(image)\n        if self.tta == "none":\n            return tensor.unsqueeze(0)\n        return torch.stack([TF.rotate(view, angle) for view in (tensor, TF.hflip(tensor)) for angle in (0, 90, 180, 270)])\n\n\nclass GeoDataset(Dataset):\n    """Lazy image decoding; IDs are preserved verbatim and never guessed."""\n    def __init__(self, frame, data_dir, split, transform):\n        if split not in {"train", "val", "test"}:\n            raise ValueError(f"Unsupported image split: {split}")\n        self.frame = frame.reset_index(drop=True).copy()\n        self.root = (Path(data_dir) / split).resolve()\n        self.transform = transform\n\n    def __len__(self):\n        return len(self.frame)\n\n    def __getitem__(self, index):\n        row = self.frame.iloc[index]\n        path = (self.root / str(row["id"])).resolve()\n        if path.parent != self.root:\n            raise ValueError("Image ID resolves outside its split directory")\n        with Image.open(path) as image:\n            tensor = self.transform(image.convert("RGB"))\n        return tensor, int(row["label"]) if "label" in self.frame else -1\n\n\ndef build_model(cfg, num_classes, pretrained=True):\n    """Build a fresh model; prediction uses pretrained=False and cannot download."""\n    name = cfg["model"]\n    dropout = float(cfg.get("dropout", 0.3))\n    if name not in MODEL_NAMES or num_classes < 2:\n        raise ValueError(f"Unsupported model or class count: {name}, {num_classes}")\n    if name == "mlp":\n        return nn.Sequential(nn.Flatten(), nn.Linear(3 * 32 * 32, 512), nn.ReLU(), nn.Dropout(dropout),\n                             nn.Linear(512, 256), nn.ReLU(), nn.Dropout(dropout), nn.Linear(256, num_classes))\n    if name.startswith("cnn_"):\n        layers, channels = [], 3\n        widths = (32, 64, 128, 256) if name == "cnn_gap" else (32, 64, 128)\n        for width in widths:\n            layers.append(nn.Conv2d(channels, width, 3, padding=1, bias=name == "cnn_plain"))\n            if name != "cnn_plain":\n                layers.append(nn.BatchNorm2d(width))\n            layers += [nn.ReLU(), nn.MaxPool2d(2)]\n            channels = width\n        grid = 1 if name == "cnn_gap" else 2\n        layers += [nn.AdaptiveAvgPool2d(grid), nn.Flatten()]\n        if name == "cnn_gap":\n            layers += [nn.Dropout(dropout), nn.Linear(channels, num_classes)]\n        else:\n            layers += [nn.Linear(channels * grid * grid, 256), nn.ReLU(), nn.Dropout(dropout), nn.Linear(256, num_classes)]\n        return nn.Sequential(*layers)\n    weights_enum = getattr(models, name.replace("resnet", "ResNet") + "_Weights")\n    weights_name = "IMAGENET1K_V2" if name == "resnet50" else "IMAGENET1K_V1"\n    weights = getattr(weights_enum, weights_name) if pretrained else None\n    backbone = getattr(models, name)(weights=weights)\n    features = backbone.fc.in_features\n    backbone.fc = nn.Identity()\n    scope = cfg.get("unfreeze", "layer4")\n    if scope not in {"head", "layer4", "layer3", "all"}:\n        raise ValueError("unfreeze must be head, layer4, layer3, or all")\n    for parameter_name, parameter in backbone.named_parameters():\n        parameter.requires_grad = scope == "all" or (scope in {"layer3", "layer4"} and parameter_name.startswith("layer4.")) or (scope == "layer3" and parameter_name.startswith("layer3."))\n    return nn.Sequential(backbone, nn.Sequential(nn.Linear(features, 512), nn.ReLU(), nn.Dropout(dropout), nn.Linear(512, num_classes)))\n\n\ndef _device(job):\n    if torch.cuda.is_available():\n        return torch.device("cuda:0")\n    if job.get("allow_cpu", False):\n        return torch.device("cpu")\n    raise RuntimeError("CUDA is required. CPU is permitted only when allow_cpu=true is explicitly set.")\n\n\ndef _loader(dataset, cfg, job, training=False):\n    workers = max(0, min(2, int(job.get("workers", 2))))\n    generator = torch.Generator().manual_seed(int(cfg["seed"]))\n    return DataLoader(dataset, batch_size=int(cfg.get("batch_size", 64)), shuffle=training,\n                      num_workers=workers, pin_memory=torch.cuda.is_available(),\n                      persistent_workers=workers > 0, worker_init_fn=_worker_seed, generator=generator)\n\n\ndef _inner_split(frame):\n    rng = np.random.default_rng(INNER_SPLIT_SEED)\n    train_indices, valid_indices = [], []\n    for _label, rows in frame.groupby("label", sort=True):\n        indices = rng.permutation(rows.index.to_numpy())\n        if len(indices) < 2:\n            raise ValueError("Every class needs at least two training rows for stratified screening")\n        count = max(1, min(len(indices) - 1, int(round(len(indices) * 0.15))))\n        valid_indices.extend(indices[:count])\n        train_indices.extend(indices[count:])\n    return frame.loc[sorted(train_indices)].reset_index(drop=True), frame.loc[sorted(valid_indices)].reset_index(drop=True)\n\n\ndef _optimizer(model, cfg):\n    learning_rate = float(cfg.get("lr", 0.001))\n    if cfg["model"].startswith("resnet"):\n        backbone = [p for p in model[0].parameters() if p.requires_grad]\n        groups = ([{"params": backbone, "lr": learning_rate * 0.1}] if backbone else [])\n        groups.append({"params": model[1].parameters(), "lr": learning_rate})\n    else:\n        groups = [{"params": model.parameters(), "lr": learning_rate}]\n    common = {"lr": learning_rate, "weight_decay": float(cfg.get("weight_decay", 0.0001))}\n    name = cfg.get("optimizer", "adamw").lower()\n    if name == "adamw":\n        return torch.optim.AdamW(groups, **common)\n    if name == "sgd":\n        return torch.optim.SGD(groups, momentum=0.9, nesterov=True, **common)\n    raise ValueError(f"Unsupported optimizer: {name}")\n\n\ndef _set_train(model, original_trainability, head_only):\n    for parameter, enabled in original_trainability:\n        parameter.requires_grad_(enabled and not head_only)\n    model.train()\n    # Frozen BatchNorm must not silently update ImageNet running statistics.\n    for module in model.modules():\n        if isinstance(module, nn.modules.batchnorm._BatchNorm) and not any(p.requires_grad for p in module.parameters()):\n            module.eval()\n\n\n@torch.inference_mode()\ndef _evaluate(model, loader, device, view_names):\n    model.eval()\n    counts, losses, total = np.zeros(len(view_names)), np.zeros(len(view_names)), 0\n    for images, labels in loader:\n        if images.ndim == 4:\n            images = images.unsqueeze(1)\n        labels = labels.to(device, non_blocking=True)\n        total += len(labels)\n        for view in range(images.shape[1]):\n            with torch.autocast(device_type=device.type, enabled=device.type == "cuda"):\n                logits = model(images[:, view].to(device, non_blocking=True))\n                loss = nn.functional.cross_entropy(logits.float(), labels, reduction="sum")\n            if not torch.isfinite(logits).all() or not torch.isfinite(loss):\n                raise FloatingPointError("Non-finite validation output")\n            counts[view] += (logits.argmax(1) == labels).sum().item()\n            losses[view] += loss.item()\n    return {name: {"accuracy": float(counts[i] / total), "loss": float(losses[i] / total)} for i, name in enumerate(view_names)}\n\n\ndef _versions():\n    return {"torch": str(torch.__version__), "torchvision": torchvision.__version__, "numpy": np.__version__, "pandas": pd.__version__}\n\n\ndef train(job):\n    """Fit an independent trial, select its best epoch, and atomically save outputs."""\n    started = time.monotonic()\n    cfg, phase = dict(job["config"]), job.get("phase", "screen")\n    if phase not in {"screen", "final"}:\n        raise ValueError("Training phase must be screen or final")\n    epochs = int(cfg.get("epochs", 10))\n    if epochs < 1 or int(cfg.get("batch_size", 64)) < 1 or _size(cfg) < 16:\n        raise ValueError("epochs/batch_size must be positive and image size at least 16")\n    seed = int(cfg["seed"])\n    seed_everything(seed)\n    device = _device(job)\n    tables = load_tables(job["data_dir"], phase=phase)\n    output = Path(job["output_dir"])\n    output.mkdir(parents=True, exist_ok=True)\n    result_path = output / "result.json"\n    if result_path.exists():\n        previous = json.loads(result_path.read_text(encoding="utf-8"))\n        if previous.get("config") == cfg and previous.get("phase") == phase and previous.get("seed") == seed and previous.get("data_signature") == tables["data_signature"] and (output / "best.pt").exists():\n            return previous\n        raise RuntimeError("Existing result does not exactly match config, phase, seed, and CSV fingerprints")\n    if any((output / name).exists() for name in ("best.pt", "best.pt.tmp", "history.json", "history.json.tmp")):\n        raise RuntimeError("Incomplete run is non-resumable; choose a new output directory")\n    if phase == "screen":\n        training_frame, valid_frame = _inner_split(tables["train"])\n        valid_dataset = GeoDataset(valid_frame, job["data_dir"], "train", _ScreenViews(cfg))\n        view_names = ("clean", "blur", "lowres", "invert")\n    else:\n        training_frame = tables["train"]\n        valid_dataset = GeoDataset(tables["val"], job["data_dir"], "val", build_transform(cfg))\n        view_names = ("clean",)\n    training = _loader(GeoDataset(training_frame, job["data_dir"], "train", build_transform(cfg, training=True)), cfg, job, True)\n    validation = _loader(valid_dataset, cfg, job)\n    model = build_model(cfg, len(tables["classes"]), pretrained=True).to(device)\n    optimizer = _optimizer(model, cfg)\n    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)\n    try:\n        scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")\n    except (AttributeError, TypeError):\n        scaler = torch.cuda.amp.GradScaler(enabled=device.type == "cuda")\n    criterion = nn.CrossEntropyLoss(label_smoothing=float(cfg.get("label_smoothing", 0.0)))\n    backbone_flags = [(p, p.requires_grad) for p in model[0].parameters()] if cfg["model"].startswith("resnet") else []\n    warmup = max(0, int(cfg.get("warmup_epochs", 1))) if backbone_flags else 0\n    history, best_metric, best_loss, best_epoch, best_views = [], -math.inf, math.inf, 0, None\n    for epoch in range(1, epochs + 1):\n        _set_train(model, backbone_flags, epoch <= warmup)\n        total, correct, loss_sum = 0, 0, 0.0\n        for images, labels in training:\n            images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)\n            optimizer.zero_grad(set_to_none=True)\n            with torch.autocast(device_type=device.type, enabled=device.type == "cuda"):\n                logits = model(images)\n                loss = criterion(logits, labels)\n            if not torch.isfinite(loss):\n                raise FloatingPointError("Non-finite training loss")\n            scaler.scale(loss).backward()\n            scaler.step(optimizer)\n            scaler.update()\n            total += len(labels)\n            correct += (logits.detach().argmax(1) == labels).sum().item()\n            loss_sum += loss.item() * len(labels)\n        views = _evaluate(model, validation, device, view_names)\n        metric = float(np.mean([value["accuracy"] for value in views.values()]))\n        eval_loss = float(np.mean([value["loss"] for value in views.values()]))\n        row = {"epoch": epoch, "train_loss": loss_sum / total, "train_accuracy": correct / total,\n               "selection_metric": metric, "evaluation": views, "learning_rates": [group["lr"] for group in optimizer.param_groups]}\n        history.append(row)\n        # Screening ties preserve the earlier epoch: selection uses the proxy alone.\n        improved = metric > best_metric or (phase == "final" and metric == best_metric and eval_loss < best_loss)\n        if improved:\n            best_metric, best_loss, best_epoch, best_views = metric, eval_loss, epoch, views\n            checkpoint = {"state_dict": model.state_dict(), "config": cfg, "num_classes": len(tables["classes"]),\n                          "classes": tables["classes"], "epoch": epoch, "selection_metric": metric,\n                          "phase": phase, "data_signature": tables["data_signature"]}\n            torch.save(checkpoint, output / "best.pt.tmp")\n            os.replace(output / "best.pt.tmp", output / "best.pt")\n        _json_write(output / "history.json", history)\n        scheduler.step()\n        print(json.dumps({"trial_id": cfg["trial_id"], "phase": phase, "epoch": epoch, "selection_metric": metric}), flush=True)\n    result = {"trial_id": cfg["trial_id"], "model": cfg["model"], "seed": seed, "phase": phase,\n              "selection_metric": best_metric, "best_epoch": best_epoch, "elapsed_seconds": time.monotonic() - started,\n              "config": cfg, "versions": _versions(), "data_signature": tables["data_signature"],\n              "checkpoint": str((output / "best.pt").resolve()), "best_evaluation": best_views,\n              "selection_metric_name": "synthetic_inner_four_view_accuracy" if phase == "screen" else "official_val_accuracy"}\n    result["inner_accuracy" if phase == "screen" else "val_accuracy"] = best_views["clean"]["accuracy"]\n    if phase == "screen":\n        result["inner_balanced_accuracy"] = best_metric\n        result["inner_split_seed"] = INNER_SPLIT_SEED\n        result["metric_note"] = "Synthetic corruption proxy on a held-out training subset; no official validation data used."\n    _json_write(result_path, result)\n    return result\n\n\n@torch.inference_mode()\ndef predict_job(job):\n    """Write ordered probabilities, averaging softmax probabilities across TTA views."""\n    checkpoint_path = Path(job["checkpoint"])\n    if job.get("checkpoint_sha256") and _sha256(checkpoint_path) != job["checkpoint_sha256"]:\n        raise ValueError("Checkpoint contents changed after prediction job was scheduled")\n    try:\n        checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)\n    except TypeError:\n        checkpoint = torch.load(checkpoint_path, map_location="cpu")\n    if checkpoint.get("phase") != "final":\n        raise ValueError("Only final-stage checkpoints may predict official val/test sets")\n    cfg = checkpoint["config"]\n    seed_everything(int(cfg["seed"]))\n    device = _device(job)\n    split, tta = job.get("split", "test"), job.get("tta", "none")\n    if split not in {"val", "test"}:\n        raise ValueError("Prediction split must be val or test")\n    tables = load_tables(job["data_dir"], phase="final")\n    if checkpoint["classes"] != tables["classes"] or checkpoint["num_classes"] != len(tables["classes"]):\n        raise ValueError("Checkpoint class mapping differs from current classes.csv")\n    for name in ("classes.csv", "train.csv", f"{split}.csv"):\n        if checkpoint.get("data_signature", {}).get(name) != tables["data_signature"][name]:\n            raise ValueError(f"Checkpoint CSV fingerprint differs: {name}")\n    model = build_model(cfg, checkpoint["num_classes"], pretrained=False)\n    model.load_state_dict(checkpoint["state_dict"], strict=True)\n    model.to(device).eval()\n    frame = tables[split]\n    loader = _loader(GeoDataset(frame, job["data_dir"], split, _PredictionViews(cfg, tta)), cfg, job)\n    batches = []\n    for images, _labels in loader:\n        probabilities = None\n        for view in range(images.shape[1]):\n            with torch.autocast(device_type=device.type, enabled=device.type == "cuda"):\n                logits = model(images[:, view].to(device, non_blocking=True))\n            if logits.ndim != 2 or logits.shape[1] != checkpoint["num_classes"] or not torch.isfinite(logits).all():\n                raise ValueError("Invalid prediction logits or class count")\n            current = logits.float().softmax(dim=1).cpu()\n            probabilities = current if probabilities is None else probabilities + current\n        batches.append((probabilities / images.shape[1]).numpy())\n    probabilities = np.concatenate(batches).astype(np.float32)\n    if not np.isfinite(probabilities).all() or not np.allclose(probabilities.sum(axis=1), 1, atol=1e-5):\n        raise ValueError("Predictions must be finite normalized probabilities")\n    prefix = Path(job.get("prefix", str(Path(job["output_dir"]) / f"{split}_{tta}")))\n    prefix.parent.mkdir(parents=True, exist_ok=True)\n    arrays = {"ids": frame["id"].to_numpy(dtype=str), "probabilities": probabilities,\n              "classes": np.asarray(checkpoint["classes"], dtype=str),\n              "class_ids": np.arange(checkpoint["num_classes"], dtype=np.int64)}\n    metadata = {"tta": tta, "split": split, "checkpoint": str(checkpoint_path.resolve()),\n                "checkpoint_sha256": _sha256(checkpoint_path), "num_classes": checkpoint["num_classes"],\n                "classes": checkpoint["classes"], "class_ids": list(range(checkpoint["num_classes"])),\n                "count": len(frame), "data_signature": tables["data_signature"], "versions": _versions()}\n    if split == "val":\n        arrays["labels"] = frame["label"].to_numpy(dtype=np.int64)\n        metadata["accuracy"] = float((probabilities.argmax(1) == arrays["labels"]).mean())\n    npz_path = Path(str(prefix) + ".npz")\n    temporary = Path(str(npz_path) + ".tmp")\n    with temporary.open("wb") as stream:\n        np.savez_compressed(stream, **arrays)\n    os.replace(temporary, npz_path)\n    _json_write(Path(str(prefix) + ".json"), metadata)\n    return metadata\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--job", required=True, type=Path)\n    args = parser.parse_args()\n    job = json.loads(args.job.read_text(encoding="utf-8-sig"))\n    result = predict_job(job) if job.get("phase") == "predict" else train(job)\n    print(json.dumps({"status": "ok", "phase": job.get("phase", "screen"),\n                      "output_dir": job["output_dir"], "selection_metric": result.get("selection_metric"),\n                      "accuracy": result.get("accuracy")}), flush=True)\n\n\nif __name__ == "__main__":\n    main()\n', 'geo_search.py': '"""Independent-GPU experiment controller. No competition data is bundled.\n\nThe notebook calls run_pipeline() on Kaggle. All fits use train only; the broad\nsearch uses a synthetic-corruption inner holdout, not the official validation.\n"""\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\nimport signal\nimport subprocess\nimport sys\nimport time\nfrom zipfile import ZipFile\n\nimport numpy as np\nimport pandas as pd\nimport torch\nimport torchvision\n\nBASE_SEED = 20261003\n\n\ndef write_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_suffix(path.suffix + \'.tmp\')\n    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding=\'utf-8\')\n    temporary.replace(path)\n\n\ndef digest(path):\n    return hashlib.sha256(Path(path).read_bytes()).hexdigest()\n\n\ndef stop_process(process):\n    """Kaggle/Linux: terminate the whole trial, including DataLoader children."""\n    try:\n        if os.name == \'posix\':\n            os.killpg(process.pid, signal.SIGTERM)\n        elif process.poll() is None:\n            process.terminate()\n        process.wait(timeout=15)\n    except ProcessLookupError:\n        return\n    except subprocess.TimeoutExpired:\n        try:\n            if os.name == \'posix\':\n                os.killpg(process.pid, signal.SIGKILL)\n            else:\n                process.kill()\n        except ProcessLookupError:\n            pass\n        process.wait()\n\n\ndef find_data_dirs(root):\n    root = Path(root)\n    if not root.exists():\n        return []\n    required = (\'train.csv\', \'val.csv\', \'test.csv\', \'classes.csv\')\n    return sorted({p.parent.resolve() for p in root.rglob(\'train.csv\')\n                   if all((p.parent / name).is_file() for name in required)\n                   and all((p.parent / name).is_dir() for name in (\'train\', \'val\', \'test\'))})\n\n\ndef prepare_data(explicit=None, input_root=\'/kaggle/input\', working=\'/kaggle/working\'):\n    """Prefer explicit path; otherwise discover one dataset or unpack one ZIP."""\n    if explicit:\n        target = Path(explicit).resolve()\n        if target not in find_data_dirs(target):\n            raise FileNotFoundError(f\'Неполный набор по DATA_DIR: {target}\')\n        return target\n    roots = [Path(input_root), Path(working)]\n    found = sorted(set(p for root in roots for p in find_data_dirs(root)))\n    if len(found) == 1:\n        return found[0]\n    if len(found) > 1:\n        raise RuntimeError(\'Несколько наборов; укажи DATA_DIR явно: \' + str(found))\n    archive_name = \'week-5-competition-geo-satellites-classification.zip\'\n    archives = sorted({p.resolve() for root in roots if root.exists()\n                       for p in root.rglob(archive_name) if p.is_file()})\n    if len(archives) != 1:\n        raise FileNotFoundError(\'Добавь данные Kaggle или укажи DATA_DIR. Архивы: \' + str(archives))\n    destination = Path(working) / \'geo_satellites_data\'\n    destination.mkdir(parents=True, exist_ok=True)\n    with ZipFile(archives[0]) as archive:\n        for item in archive.infolist():\n            if not (destination / item.filename).resolve().is_relative_to(destination.resolve()):\n                raise ValueError(\'Небезопасный путь в ZIP: \' + item.filename)\n        archive.extractall(destination)\n    found = find_data_dirs(destination)\n    if len(found) != 1:\n        raise RuntimeError(\'Проверь содержимое распакованного архива: \' + str(found))\n    return found[0]\n\n\ndef screen_plan(budget=\'extended\'):\n    base = dict(seed=BASE_SEED, epochs=8, batch_size=32, lr=1e-3,\n                weight_decay=1e-3, dropout=0.3, augment=\'robust\', unfreeze=\'layer3\',\n                optimizer=\'adamw\', label_smoothing=0.05, size=160)\n    variants = [\n        (\'mlp\', \'mlp\', dict(size=32, epochs=16, batch_size=64, dropout=0.2,\n                           augment=\'standard\', label_smoothing=0.0)),\n        (\'cnn_plain\', \'cnn_plain\', dict(size=96, epochs=16, augment=\'standard\', dropout=0.0)),\n        (\'cnn_bn\', \'cnn_bn\', dict(size=96, epochs=16, augment=\'standard\', dropout=0.0)),\n        (\'cnn_gap_standard\', \'cnn_gap\', dict(size=128, epochs=20, augment=\'standard\')),\n        (\'cnn_gap_robust\', \'cnn_gap\', dict(size=128, epochs=20)),\n        (\'r18_standard\', \'resnet18\', dict(augment=\'standard\')),\n        (\'r18_robust\', \'resnet18\', dict()),\n        (\'r34_robust\', \'resnet34\', dict(lr=7e-4)),\n        (\'r50_robust\', \'resnet50\', dict(batch_size=24)),\n    ]\n    if budget == \'extended\':\n        variants += [\n            (\'r18_layer4\', \'resnet18\', dict(unfreeze=\'layer4\')),\n            (\'r34_all\', \'resnet34\', dict(unfreeze=\'all\', lr=5e-4)),\n        ]\n    elif budget != \'moderate\':\n        raise ValueError(\'budget must be moderate or extended\')\n    return [dict(base, **{\'trial_id\': name, \'model\': model, **changes})\n            for name, model, changes in variants]\n\n\ndef ranked_unique(results, count, predicate=lambda r: True):\n    picked, seen = [], set()\n    for result in sorted(results, key=lambda r: (-r[\'selection_metric\'], r[\'trial_id\'])):\n        family = result[\'config\'][\'model\']\n        if predicate(result) and family not in seen:\n            picked.append(result)\n            seen.add(family)\n            if len(picked) == count:\n                break\n    return picked\n\n\ndef refinement_plan(results, budget):\n    leaders = ranked_unique(results, 2, lambda r: r[\'config\'][\'model\'].startswith(\'resnet\'))\n    configs = []\n    for rank, leader in enumerate(leaders):\n        original = leader[\'config\']\n        changes = [(\'lr_half\', dict(lr=original[\'lr\'] / 2)),\n                   (\'lr_double\', dict(lr=original[\'lr\'] * 2))]\n        if budget == \'extended\':\n            changes += [(\'sgd\', dict(optimizer=\'sgd\', lr=0.02, weight_decay=1e-4))]\n        for name, change in changes:\n            configs.append(dict(original, **{\'trial_id\': f\'refine_{rank}_{name}\', **change}))\n    return configs\n\n\ndef final_plan(results, budget):\n    chosen = ranked_unique(results, 2, lambda r: r[\'config\'][\'model\'].startswith(\'resnet\'))\n    chosen += ranked_unique(results, 1, lambda r: r[\'config\'][\'model\'].startswith(\'cnn\'))\n    chosen += ranked_unique(results, 1, lambda r: r[\'config\'][\'model\'] == \'mlp\')\n    configs = []\n    for leader in chosen:\n        cfg = dict(leader[\'config\'])\n        cfg[\'trial_id\'] = \'final_\' + cfg[\'trial_id\']\n        if cfg[\'model\'].startswith(\'resnet\'):\n            cfg.update(size=224, epochs=24 if budget == \'extended\' else 16,\n                       batch_size=24 if cfg[\'model\'] == \'resnet50\' else 32)\n        else:\n            cfg[\'epochs\'] = 45 if budget == \'extended\' else 30\n        configs.append(cfg)\n    return configs\n\n\nclass GPUPool:\n    """One child process per GPU; no DataParallel and no shared model state."""\n    def __init__(self, data_dir, output_dir, workers=2):\n        self.data_dir = str(Path(data_dir).resolve())\n        self.output_dir = Path(output_dir).resolve()\n        self.worker = Path(__file__).with_name(\'geo_worker.py\').resolve()\n        self.workers = workers\n        count = torch.cuda.device_count()\n        if count < 1:\n            raise RuntimeError(\'Включи GPU в Kaggle Settings → Accelerator. CPU-поиск отключён.\')\n        visible = os.environ.get(\'CUDA_VISIBLE_DEVICES\')\n        self.gpus = visible.split(\',\')[:min(count, 2)] if visible else [str(i) for i in range(min(count, 2))]\n        self.output_dir.mkdir(parents=True, exist_ok=True)\n        context = {\n            \'data_dir\': self.data_dir,\n            \'csv_hashes\': {name: digest(Path(data_dir) / name) for name in\n                           (\'train.csv\', \'val.csv\', \'test.csv\', \'classes.csv\', \'sample_submission_example.csv\')\n                           if (Path(data_dir) / name).is_file()},\n            \'worker_sha256\': digest(self.worker), \'controller_sha256\': digest(__file__),\n            \'torch\': torch.__version__, \'torchvision\': torchvision.__version__,\n            \'gpu_names\': [torch.cuda.get_device_name(i) for i in range(len(self.gpus))],\n            \'workers\': workers,\n        }\n        manifest = self.output_dir / \'run_context.json\'\n        if manifest.exists() and json.loads(manifest.read_text(encoding=\'utf-8\')) != context:\n            raise RuntimeError(\'Изменились данные, код или окружение. Выбери новый RUN_NAME.\')\n        write_json(manifest, context)\n        print(\'GPU для независимых запусков:\', self.gpus, flush=True)\n\n    def run(self, stage, jobs):\n        pending, completed = [], []\n        for identity, payload in jobs:\n            base = self.output_dir / stage / identity\n            job_signature = dict(payload, data_dir=self.data_dir, workers=self.workers, allow_cpu=False)\n            signature_file = base / \'job_signature.json\'\n            if signature_file.exists():\n                old = json.loads(signature_file.read_text(encoding=\'utf-8\'))\n                if old != job_signature:\n                    raise RuntimeError(f\'Настройки {identity} изменились: выбери другой RUN_NAME.\')\n            write_json(signature_file, job_signature)\n            done = base / \'completed.json\'\n            if done.exists():\n                previous = json.loads(done.read_text(encoding=\'utf-8\'))\n                if payload[\'phase\'] != \'predict\' and not (Path(previous[\'_directory\']) / \'best.pt\').exists():\n                    raise FileNotFoundError(\'Checkpoint удалён: \' + previous[\'_directory\'])\n                if payload[\'phase\'] != \'predict\' and digest(Path(previous[\'_directory\']) / \'best.pt\') != previous.get(\'_checkpoint_sha256\'):\n                    raise RuntimeError(\'Checkpoint изменён после обучения: \' + previous[\'_directory\'])\n                if payload[\'phase\'] == \'predict\' and not Path(previous[\'_prefix\'] + \'.npz\').exists():\n                    raise FileNotFoundError(\'Прогнозы удалены: \' + previous[\'_prefix\'])\n                if payload[\'phase\'] == \'predict\' and digest(previous[\'_prefix\'] + \'.npz\') != previous.get(\'_npz_sha256\'):\n                    raise RuntimeError(\'Сохранённые прогнозы изменены: \' + previous[\'_prefix\'])\n                completed.append(previous)\n                print(f\'[{stage}] reuse {identity}\', flush=True)\n                continue\n            attempt = 1\n            while (base / f\'attempt_{attempt:03d}\').exists():\n                attempt += 1\n            directory = base / f\'attempt_{attempt:03d}\'\n            directory.mkdir(parents=True)\n            job = dict(job_signature, output_dir=str(directory))\n            if payload[\'phase\'] == \'predict\':\n                job[\'prefix\'] = str(directory / \'predictions\')\n            path = directory / \'job.json\'\n            write_json(path, job)\n            pending.append((identity, job, path, done))\n        active = {}\n        last_status = time.monotonic()\n        try:\n            while pending or active:\n                for gpu in self.gpus:\n                    if gpu in active or not pending:\n                        continue\n                    identity, job, path, done = pending.pop(0)\n                    logfile = Path(job[\'output_dir\']) / \'stdout.log\'\n                    handle = logfile.open(\'w\', encoding=\'utf-8\')\n                    env = dict(os.environ, CUDA_VISIBLE_DEVICES=gpu, PYTHONUNBUFFERED=\'1\',\n                               OMP_NUM_THREADS=\'2\', MKL_NUM_THREADS=\'2\',\n                               CUBLAS_WORKSPACE_CONFIG=\':4096:8\')\n                    try:\n                        process = subprocess.Popen([sys.executable, str(self.worker), \'--job\', str(path)],\n                                                   stdout=handle, stderr=subprocess.STDOUT, env=env,\n                                                   start_new_session=(os.name == \'posix\'))\n                    except BaseException:\n                        handle.close()\n                        raise\n                    active[gpu] = (process, handle, identity, job, done, logfile)\n                    print(f\'[{stage}] GPU {gpu}: {identity}; log={logfile}\', flush=True)\n                for gpu in list(active):\n                    process, handle, identity, job, done, logfile = active[gpu]\n                    status = process.poll()\n                    if status is None:\n                        continue\n                    handle.close()\n                    del active[gpu]\n                    if status:\n                        stop_process(process)\n                        tail = \'\\n\'.join(logfile.read_text(encoding=\'utf-8\', errors=\'replace\').splitlines()[-35:])\n                        raise RuntimeError(f\'{identity} завершился с кодом {status}:\\n{tail}\')\n                    directory = Path(job[\'output_dir\'])\n                    if job[\'phase\'] == \'predict\':\n                        result = json.loads(Path(job[\'prefix\'] + \'.json\').read_text(encoding=\'utf-8\'))\n                        result[\'_prefix\'] = job[\'prefix\']\n                        result[\'_npz_sha256\'] = digest(job[\'prefix\'] + \'.npz\')\n                    else:\n                        result = json.loads((directory / \'result.json\').read_text(encoding=\'utf-8\'))\n                        result[\'_checkpoint_sha256\'] = digest(directory / \'best.pt\')\n                        if result.get(\'phase\') != job[\'phase\'] or result.get(\'config\') != job[\'config\'] or result.get(\'trial_id\') != job[\'config\'][\'trial_id\'] or not np.isfinite(result.get(\'selection_metric\', np.nan)):\n                            raise ValueError(\'Неверный результат worker: \' + identity)\n                    if job[\'phase\'] == \'predict\' and (result.get(\'split\') != job[\'split\'] or result.get(\'tta\') != job[\'tta\'] or result.get(\'checkpoint_sha256\') != job[\'checkpoint_sha256\']):\n                        raise ValueError(\'Неверные метаданные предсказаний: \' + identity)\n                    result.update(_directory=str(directory), _identity=identity)\n                    write_json(done, result)\n                    completed.append(result)\n                    print(f\'[{stage}] DONE {identity}: {result.get("selection_metric", result.get("accuracy", "OK"))}\', flush=True)\n                if time.monotonic() - last_status > 60:\n                    for gpu, (_, _, identity, _, _, logfile) in active.items():\n                        lines = logfile.read_text(encoding=\'utf-8\', errors=\'replace\').splitlines()\n                        print(f\'GPU {gpu}: {identity}: {lines[-1] if lines else "starting"}\', flush=True)\n                    last_status = time.monotonic()\n                if active:\n                    time.sleep(3)\n        except BaseException:\n            for process, handle, *_ in active.values():\n                try:\n                    stop_process(process)\n                finally:\n                    handle.close()\n            raise\n        return sorted(completed, key=lambda r: r[\'_identity\'])\n\n\ndef train_jobs(configs, phase):\n    return [(cfg[\'trial_id\'], {\'phase\': phase, \'config\': cfg}) for cfg in configs]\n\n\ndef run_training(pool, budget=\'extended\'):\n    configs = screen_plan(budget)\n    print(f\'Screen: {len(configs)} моделей, сумма эпох = {sum(c["epochs"] for c in configs)}\')\n    screening = pool.run(\'screen\', train_jobs(configs, \'screen\'))\n    refinement = pool.run(\'refine\', train_jobs(refinement_plan(screening, budget), \'screen\'))\n    finalists = pool.run(\'final\', train_jobs(final_plan(screening + refinement, budget), \'final\'))\n    # Pick TWO completed configurations, then repeat each on predeclared seeds.\n    leaders = sorted(finalists, key=lambda r: (-r[\'selection_metric\'], r[\'trial_id\']))[:2]\n    seeds = (BASE_SEED + 101, BASE_SEED + 202) if budget == \'extended\' else (BASE_SEED + 101,)\n    repeats = []\n    for leader in leaders:\n        for seed in seeds:\n            cfg = dict(leader[\'config\'], seed=seed,\n                       trial_id=leader[\'trial_id\'] + f\'_seed{seed}\')\n            repeats.append(cfg)\n    confirmed = pool.run(\'confirm\', train_jobs(repeats, \'final\'))\n    all_results = screening + refinement + finalists + confirmed\n    rows = [{key: r.get(key) for key in (\'trial_id\', \'model\', \'phase\', \'selection_metric\',\n                                        \'best_epoch\', \'elapsed_seconds\')}\n            | {\'seed\': r[\'config\'][\'seed\'], \'config\': json.dumps(r[\'config\'], sort_keys=True)}\n            for r in all_results]\n    pd.DataFrame(rows).to_csv(pool.output_dir / \'experiments.csv\', index=False)\n    # Never choose the luckiest seed. All declared repeats enter their group.\n    groups = []\n    for leader in leaders:\n        group = [leader] + [r for r in confirmed if r[\'trial_id\'].startswith(leader[\'trial_id\'] + \'_seed\')]\n        groups.append(group)\n    write_json(pool.output_dir / \'training_summary.json\', {\'finalists\': finalists, \'groups\': groups})\n    return finalists, groups\n\n\ndef predict_jobs(records, split, tta):\n    return [(r[\'trial_id\'] + \'_\' + split + \'_\' + tta,\n             {\'phase\': \'predict\', \'config\': r[\'config\'], \'checkpoint\': str(Path(r[\'_directory\']) / \'best.pt\'),\n              \'checkpoint_sha256\': digest(Path(r[\'_directory\']) / \'best.pt\'),\n              \'split\': split, \'tta\': tta}) for r in records]\n\n\ndef aligned_mean(predictions):\n    """Average probabilities with strict id/label/column alignment checks."""\n    arrays = [np.load(p[\'_prefix\'] + \'.npz\', allow_pickle=False) for p in predictions]\n    try:\n        ids = arrays[0][\'ids\'].astype(str)\n        if ids.ndim != 1 or len(np.unique(ids)) != len(ids):\n            raise ValueError(\'Ожидаются уникальные одномерные id.\')\n        labels = arrays[0][\'labels\'].copy() if \'labels\' in arrays[0].files else None\n        class_ids = arrays[0][\'class_ids\']\n        classes = arrays[0][\'classes\'].astype(str)\n        shape = arrays[0][\'probabilities\'].shape\n        if len(shape) != 2 or shape[0] != len(ids) or shape[1] != len(classes) or not np.array_equal(class_ids, np.arange(shape[1])):\n            raise ValueError(\'Форма прогнозов не соответствует id или classes.\')\n        total = np.zeros(shape, dtype=np.float64)\n        for values in arrays:\n            if not np.array_equal(ids, values[\'ids\'].astype(str)):\n                raise ValueError(\'Порядок id отличается между прогнозами.\')\n            if (\'labels\' in values.files) != (labels is not None):\n                raise ValueError(\'Смешаны val и test прогнозы.\')\n            if labels is not None and not np.array_equal(labels, values[\'labels\']):\n                raise ValueError(\'Метки отличаются между прогнозами.\')\n            if not np.array_equal(class_ids, values[\'class_ids\']) or not np.array_equal(classes, values[\'classes\'].astype(str)):\n                raise ValueError(\'Порядок классов отличается между моделями.\')\n            prob = values[\'probabilities\']\n            if prob.shape != shape or not np.isfinite(prob).all():\n                raise ValueError(\'Некорректные вероятности.\')\n            if (prob < 0).any() or not np.allclose(prob.sum(axis=1), 1, atol=1e-4):\n                raise ValueError(\'Ожидаются нормированные softmax-вероятности.\')\n            total += prob\n        return ids, (total / len(arrays)).astype(np.float32), labels\n    finally:\n        for values in arrays:\n            values.close()\n\n\ndef select_recipe(pool, finalists, groups):\n    rows, recipes, prediction_cache = [], [], {}\n    # Include every full-train finalist as a single-model/no-TTA candidate.\n    base_predictions = pool.run(\'predict_val\', predict_jobs(finalists, \'val\', \'none\'))\n    prediction_cache.update({p[\'_identity\']: p for p in base_predictions})\n    def add_recipe(name, members, tta):\n        keys = [r[\'trial_id\'] + \'_val_\' + tta for r in members]\n        missing = [r for r, key in zip(members, keys) if key not in prediction_cache]\n        if missing:\n            new = pool.run(\'predict_val\', predict_jobs(missing, \'val\', tta))\n            prediction_cache.update({p[\'_identity\']: p for p in new})\n        ids, probabilities, labels = aligned_mean([prediction_cache[k] for k in keys])\n        correct = int((probabilities.argmax(1) == labels).sum())\n        complexity = len(members) * {\'none\': 1, \'d4\': 8, \'fivecrop\': 5}[tta]\n        row = dict(recipe=name, val_accuracy=correct / len(labels), correct=correct,\n                   count=len(labels), forward_views=complexity)\n        rows.append(row)\n        recipes.append(dict(name=name, members=members, tta=tta, **row))\n        return ids, probabilities, labels\n    for r in finalists:\n        add_recipe(\'single_\' + r[\'trial_id\'], [r], \'none\')\n    for i, group in enumerate(groups):\n        add_recipe(f\'seeds_group{i}\', group, \'none\')\n    all_members = [r for group in groups for r in group]\n    add_recipe(\'equal_ensemble\', all_members, \'none\')\n    add_recipe(\'equal_ensemble_d4\', all_members, \'d4\')\n    add_recipe(\'equal_ensemble_fivecrop\', all_members, \'fivecrop\')\n    table = pd.DataFrame(rows).sort_values([\'val_accuracy\', \'forward_views\'], ascending=[False, True])\n    table.to_csv(pool.output_dir / \'validation_recipes.csv\', index=False)\n    winner = sorted(recipes, key=lambda r: (-r[\'val_accuracy\'], r[\'forward_views\']))[0]\n    write_json(pool.output_dir / \'selected_recipe.json\', winner)\n    keys = [r[\'trial_id\'] + \'_val_\' + winner[\'tta\'] for r in winner[\'members\']]\n    ids, probabilities, labels = aligned_mean([prediction_cache[k] for k in keys])\n    np.savez_compressed(pool.output_dir / \'selected_validation.npz\', ids=ids,\n                        probabilities=probabilities, labels=labels)\n    print(table.to_string(index=False))\n    print(\'Выбрано:\', winner[\'name\'], \'| validation accuracy:\', winner[\'val_accuracy\'])\n    if winner[\'val_accuracy\'] <= .65:\n        print(\'ВНИМАНИЕ: validation пока не превышает 65%. Улучшение не подтверждено.\')\n    return winner\n\n\ndef make_submission(pool, recipe, filename=\'submission_geo.csv\'):\n    # The recipe is already frozen. Test is used only for inference/export.\n    predictions = pool.run(\'predict_test\', predict_jobs(recipe[\'members\'], \'test\', recipe[\'tta\']))\n    ids, probabilities, _ = aligned_mean(predictions)\n    test = pd.read_csv(Path(pool.data_dir) / \'test.csv\', dtype={\'id\': str})\n    if not np.array_equal(ids, test[\'id\'].to_numpy(dtype=str)) or not test.id.is_unique:\n        raise ValueError(\'Нарушен порядок или уникальность test id.\')\n    submission = pd.DataFrame({\'id\': ids, \'label\': probabilities.argmax(1)})\n    template_path = Path(pool.data_dir) / \'sample_submission_example.csv\'\n    if template_path.exists():\n        sample = pd.read_csv(template_path, dtype={\'id\': str})\n        if list(sample.columns) != [\'id\', \'label\']:\n            raise ValueError(\'Проверь формат sample_submission_example.csv.\')\n        if len(sample) == len(submission) and sample.id.is_unique and set(sample.id) == set(ids):\n            submission = submission.set_index(\'id\').loc[sample.id].reset_index()\n        else:\n            print(\'sample_submission_example — пример формата, не полный test; сохранён порядок test.csv.\')\n    if not submission.label.between(0, probabilities.shape[1] - 1).all():\n        raise ValueError(\'Неверные индексы классов.\')\n    if Path(filename).name != filename or not filename.endswith(\'.csv\'):\n        raise ValueError(\'filename должен быть именем CSV без каталогов.\')\n    destination = pool.output_dir / filename\n    submission.to_csv(destination, index=False)\n    print(\'Готовый submission:\', destination, \'| строк:\', len(submission))\n    return destination\n\n\ndef run_pipeline(data_dir, output_dir, budget=\'extended\', workers=2):\n    pool = GPUPool(data_dir, output_dir, workers)\n    finalists, groups = run_training(pool, budget)\n    recipe = select_recipe(pool, finalists, groups)\n    return make_submission(pool, recipe)\n', 'geo_v7_45_worker.py': '"""Bounded final training, separate prediction, and FP32 last-full-epoch EMA.\n\nOnly train and explicitly selected labeled validation rows are fitted. Test is\nread solely by prediction/cache-for-prediction jobs, never by a training job.\nScreening fits train only. Final fitting uses supplied labeled train + val.\nTest metadata/images are inaccessible from a training job. Screening is not\npart of the separately measured final-fitting window.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport copy\nimport hashlib\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport time\n\nos.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")\n\nimport numpy as np\nfrom PIL import Image\nimport torch\nfrom torch import nn\nfrom torch.utils.data import Dataset, ConcatDataset\nfrom torchvision import models\nfrom torchvision.transforms import v2\nfrom torchvision.transforms import functional as TF\n\nimport geo_worker as base\n\n\nPROTOCOL = "v7_45_trainval_separate_predict"\nMODELS = ("resnet152", "resnet101", "resnet50", "inception_v3", "googlenet", "vgg16_bn")\nSEEDS = (20261003, 20261104)\nMEAN, STD = base.MEAN, base.STD\n\n\nclass BudgetExpired(RuntimeError):\n    pass\n\n\ndef check_deadline(job):\n    deadline = float(job.get("deadline_unix", job.get("deadline_epoch", math.inf)))\n    if time.time() >= deadline:\n        raise BudgetExpired("Worker training deadline expired")\n\n\ndef config_for_model(model, seed, trial_id=None, crop_scale=(.55, 1.0)):\n    cfg = dict(model=model, seed=int(seed), trial_id=trial_id or f"v7_45_{model}_seed{seed}",\n               epochs=14, minimum_epochs=3, size=299 if model == "inception_v3" else 224,\n               batch_size=16 if model == "resnet152" else 24, val_repeat=2,\n               lr_backbone=.0002, lr_head=.002, weight_decay=.05, dropout=.3,\n               label_smoothing=.1, mix_p=.3, mixup_alpha=.4, cutmix_alpha=1.0,\n               ema_decay=.999, grad_clip=2.0, warmup_epochs=1,\n               crop_scale=list(crop_scale), freeze_early=model.startswith("resnet"),\n               inception_transform_input=model == "inception_v3",\n               augment="torchvision_v2_corruptions", optimizer="adamw")\n    validate_config(cfg)\n    return cfg\n\n\ndef validate_config(cfg):\n    if cfg.get("model") not in MODELS:\n        raise ValueError("Only the six explicitly supported torchvision CNN backbones are permitted")\n    for name in ("epochs", "minimum_epochs", "size", "batch_size", "val_repeat", "warmup_epochs"):\n        value = cfg.get(name)\n        if isinstance(value, bool) or not isinstance(value, int) or value <= 0:\n            raise ValueError(f"{name} must be a positive integer")\n    if cfg["size"] < 75 or (cfg["model"] == "inception_v3" and cfg["size"] < 299):\n        raise ValueError("Inputs must support the selected torchvision backbone")\n    if cfg.get("optimizer") != "adamw" or cfg.get("augment") != "torchvision_v2_corruptions":\n        raise ValueError("Fixed AdamW/official torchvision augmentation protocol required")\n    if cfg["minimum_epochs"] > cfg["epochs"]:\n        raise ValueError("minimum_epochs must not exceed epochs")\n    for name in ("lr_backbone", "lr_head", "weight_decay", "dropout", "label_smoothing",\n                 "mix_p", "ema_decay", "grad_clip", "mixup_alpha", "cutmix_alpha"):\n        value = cfg.get(name)\n        if isinstance(value, bool) or not isinstance(value, (int, float)) or not math.isfinite(value):\n            raise ValueError(f"{name} must be finite")\n    if cfg["lr_backbone"] <= 0 or cfg["lr_head"] <= 0 or cfg["grad_clip"] <= 0:\n        raise ValueError("Positive learning rates and gradient limit are required")\n    if cfg["weight_decay"] < 0 or cfg["mixup_alpha"] <= 0 or cfg["cutmix_alpha"] <= 0:\n        raise ValueError("Invalid decay or mixing alpha")\n    for name in ("dropout", "label_smoothing", "mix_p", "ema_decay"):\n        if not 0 <= cfg[name] < 1:\n            raise ValueError(f"{name} must lie in [0,1)")\n    scale = cfg.get("crop_scale")\n    if not isinstance(scale, (tuple, list)) or len(scale) != 2 or not 0 < scale[0] <= scale[1] <= 1:\n        raise ValueError("crop_scale must be a pair within (0,1]")\n    if not isinstance(cfg.get("freeze_early"), bool):\n        raise ValueError("freeze_early must be explicit")\n    if cfg["model"] == "inception_v3" and cfg["freeze_early"]:\n        raise ValueError("This independent Inception configuration does not freeze early layers")\n    if cfg.get("inception_transform_input") is not (cfg["model"] == "inception_v3"):\n        raise ValueError("Inception must retain its official pretrained internal input convention")\n\n\ndef load_metadata(data_dir, include_test=False):\n    """Training calls never open test.csv, test images or test cache arrays."""\n    root = Path(data_dir)\n    classes = base._read_csv(root / "classes.csv", ["label", "class_name"]).sort_values("label")\n    if classes["label"].tolist() != list(range(len(classes))) or len(classes) < 2:\n        raise ValueError("Class labels must be contiguous and contain at least two classes")\n    if classes["class_name"].duplicated().any() or (classes["class_name"].str.strip() == "").any():\n        raise ValueError("Class names must be unique and nonempty")\n    result = {"classes": classes["class_name"].tolist(), "data_signature": {}}\n    names = ["train", "val"] + (["test"] if include_test else [])\n    for split in names:\n        path = root / f"{split}.csv"\n        frame = base._read_csv(path, ["id"] if split == "test" else ["id", "label"])\n        if "label" in frame and not frame["label"].between(0, len(classes) - 1).all():\n            raise ValueError(f"{split}: label outside class range")\n        result[split] = frame\n        result["data_signature"][path.name] = base._sha256(path)\n    result["data_signature"]["classes.csv"] = base._sha256(root / "classes.csv")\n    return result\n\n\ndef _indices(values, count, name):\n    if not isinstance(values, (tuple, list)):\n        raise ValueError(f"{name} must be an explicit sequence")\n    if any(isinstance(x, bool) or not isinstance(x, (int, np.integer)) for x in values):\n        raise ValueError(f"{name} must contain integer indices")\n    values = [int(x) for x in values]\n    if len(set(values)) != len(values) or any(x < 0 or x >= count for x in values):\n        raise ValueError(f"{name}: duplicate or out-of-range validation index")\n    return values\n\n\ndef validate_partition(job, count):\n    fitted = _indices(job.get("train_val_indices", list(range(count)) if job.get("phase") == "final" else []), count, "train_val_indices")\n    heldout = _indices(job.get("holdout_val_indices", list(range(count)) if job.get("phase") == "screen" else []), count, "holdout_val_indices")\n    if set(fitted).intersection(heldout):\n        raise ValueError("Held-out validation rows must never be fitted")\n    if set(fitted).union(heldout) != set(range(count)):\n        raise ValueError("Fitted and held-out indices must partition all validation rows")\n    if job.get("phase") == "screen":\n        if fitted or len(heldout) != count:\n            raise ValueError("Screening fits train only; all val rows must stay held out")\n    elif job.get("phase") == "final":\n        if heldout or len(fitted) != count:\n            raise ValueError("Final fitting uses every labeled validation row and no holdout")\n    else:\n        raise ValueError("Training phase must be screen or final")\n    return fitted, heldout\n\n\ndef job_signature(job):\n    content = {key: job.get(key) for key in ("config", "phase", "min_epochs", "train_val_indices", "holdout_val_indices",\n              "split", "indices", "tta", "checkpoint_sha256", "data_signature")}\n    return hashlib.sha256(json.dumps(content, sort_keys=True, ensure_ascii=False, allow_nan=False).encode()).hexdigest()\n\n\ndef build_cache(data_dir, cache_dir, side=320, include_test=False, deadline_unix=math.inf):\n    """Deterministic per-image resizing, not dataset-fitted preprocessing.\n\n    The controller must call include_test=True only after the recipe is frozen.\n    Cache arrays contain private images and MUST NOT enter recovery archives.\n    """\n    if not isinstance(side, int) or side < 299:\n        raise ValueError("Cache side must be at least 299")\n    tables = load_metadata(data_dir, include_test=include_test)\n    root = Path(cache_dir)\n    root.mkdir(parents=True, exist_ok=True)\n    manifest_path = root / "cache_manifest.json"\n    manifest = json.loads(manifest_path.read_text(encoding="utf-8")) if manifest_path.exists() else {\n        "protocol": PROTOCOL, "side": side, "splits": {}, "data_signature": {}}\n    if manifest.get("protocol") != PROTOCOL or manifest.get("side") != side:\n        raise ValueError("Existing image cache protocol/size differs")\n    for key, value in tables["data_signature"].items():\n        if key in manifest["data_signature"] and manifest["data_signature"][key] != value:\n            raise ValueError("Existing image cache dataset differs")\n    splits = ["train", "val"] + (["test"] if include_test else [])\n    for split in splits:\n        check_deadline({"deadline_unix": deadline_unix})\n        target = root / f"{split}.npy"\n        shape = (len(tables[split]), 3, side, side)\n        if target.exists() and manifest["splits"].get(split, {}).get("shape") == list(shape):\n            existing = np.load(target, mmap_mode="r", allow_pickle=False)\n            if existing.shape != shape or existing.dtype != np.uint8:\n                raise ValueError("Existing cache array shape/dtype mismatch")\n            del existing\n            continue\n        temporary = target.with_name(target.name + ".tmp")\n        array = np.lib.format.open_memmap(temporary, mode="w+", dtype=np.uint8, shape=shape)\n        try:\n            for index, image_id in enumerate(tables[split]["id"]):\n                check_deadline({"deadline_unix": deadline_unix})\n                with Image.open(Path(data_dir) / split / image_id) as image:\n                    resized = TF.resize(image.convert("RGB"), [side, side], antialias=True)\n                    array[index] = np.asarray(resized, dtype=np.uint8).transpose(2, 0, 1)\n            array.flush()\n        finally:\n            del array\n        os.replace(temporary, target)\n        manifest["splits"][split] = {"shape": list(shape), "dtype": "uint8",\n            "table_sha256": tables["data_signature"][f"{split}.csv"]}\n        manifest["data_signature"].update(tables["data_signature"])\n        base._json_write(manifest_path, manifest)\n    return manifest\n\n\ndef build_transform(cfg, training=False):\n    size = int(cfg["size"])\n    if not training:\n        return v2.Compose([v2.ToImage(), v2.Resize((size, size), antialias=True),\n                           v2.ToDtype(torch.float32, scale=True), v2.Normalize(MEAN, STD)])\n    if not hasattr(v2, "JPEG") or not hasattr(v2, "GaussianNoise"):\n        raise RuntimeError("torchvision v2 JPEG/GaussianNoise are required (official 0.22 or later)")\n    return v2.Compose([\n        v2.ToImage(), v2.RandomResizedCrop((size, size), scale=tuple(cfg["crop_scale"]),\n                                         ratio=(.9, 1.1), antialias=True),\n        v2.RandomHorizontalFlip(), v2.RandomVerticalFlip(),\n        v2.RandomChoice([v2.RandomRotation((angle, angle)) for angle in (0, 90, 180, 270)]),\n        v2.ColorJitter(.2, .2, .2, .02),\n        v2.RandomApply([v2.GaussianBlur(5, sigma=(.2, 1.5))], p=.25),\n        v2.RandomApply([v2.JPEG(quality=(8, 45))], p=.30),\n        v2.RandomApply([v2.Resize((size // 2, size // 2), antialias=True),\n                        v2.Resize((size, size), antialias=True)], p=.25),\n        v2.RandomGrayscale(p=.10), v2.ToDtype(torch.float32, scale=True),\n        v2.RandomApply([v2.GaussianNoise(mean=0., sigma=.03, clip=True)], p=.15),\n        v2.Normalize(MEAN, STD)])\n\n\nclass ImageDataset(Dataset):\n    """Split-local rows, optional read-only memory map; never infer filename labels."""\n    def __init__(self, tables, data_dir, split, transform, indices=None, cache_dir=None):\n        if split not in {"train", "val", "test"}:\n            raise ValueError("Unknown split")\n        self.frame = tables[split]\n        self.indices = list(range(len(self.frame))) if indices is None else list(indices)\n        self.root = (Path(data_dir) / split).resolve()\n        self.split, self.transform = split, transform\n        self.cache_path, self.cache = None, None\n        if cache_dir:\n            cache_root = Path(cache_dir)\n            manifest = json.loads((cache_root / "cache_manifest.json").read_text(encoding="utf-8"))\n            if manifest.get("protocol") != PROTOCOL:\n                raise ValueError("Image cache protocol differs")\n            for key in ("classes.csv", f"{split}.csv"):\n                if manifest.get("data_signature", {}).get(key) != tables["data_signature"][key]:\n                    raise ValueError("Image cache metadata differs")\n            self.cache_path = cache_root / f"{split}.npy"\n            array = np.load(self.cache_path, mmap_mode="r", allow_pickle=False)\n            expected = tuple(manifest["splits"][split]["shape"])\n            if array.shape != expected or array.dtype != np.uint8 or array.shape[0] != len(self.frame):\n                raise ValueError("Image cache array shape/dtype differs")\n            del array\n\n    def __len__(self):\n        return len(self.indices)\n\n    def __getitem__(self, index):\n        original = self.indices[index]\n        row = self.frame.iloc[original]\n        if self.cache_path:\n            if self.cache is None:\n                self.cache = np.load(self.cache_path, mmap_mode="r", allow_pickle=False)\n            # A private contiguous writable copy is necessary for torchvision tensor ops.\n            image = torch.from_numpy(np.array(self.cache[original], copy=True))\n        else:\n            path = (self.root / str(row["id"])).resolve()\n            if path.parent != self.root:\n                raise ValueError("Image ID resolves outside split directory")\n            with Image.open(path) as source:\n                image = source.convert("RGB")\n                image = self.transform(image)\n            return image, int(row["label"]) if "label" in self.frame else -1\n        return self.transform(image), int(row["label"]) if "label" in self.frame else -1\n\n\ndef build_model(cfg, num_classes, pretrained=True):\n    validate_config(cfg)\n    if isinstance(num_classes, bool) or not isinstance(num_classes, int) or num_classes < 2:\n        raise ValueError("At least two classes required")\n    name = cfg["model"]\n    if name.startswith("resnet"):\n        enum = {"resnet152": models.ResNet152_Weights, "resnet101": models.ResNet101_Weights,\n                "resnet50": models.ResNet50_Weights}[name]\n        network = getattr(models, name)(weights=enum.IMAGENET1K_V2 if pretrained else None)\n        width = network.fc.in_features\n        network.fc = nn.Identity()\n        if cfg["freeze_early"]:\n            for block in (network.conv1, network.bn1, network.layer1, network.layer2):\n                block.requires_grad_(False)\n    elif name == "inception_v3":\n        weights = models.Inception_V3_Weights.IMAGENET1K_V1 if pretrained else None\n        network = models.inception_v3(weights=weights, aux_logits=bool(pretrained),\n                                      transform_input=cfg["inception_transform_input"], init_weights=False)\n        network.aux_logits, network.AuxLogits = False, None\n        width = network.fc.in_features\n        network.fc, network.dropout = nn.Identity(), nn.Identity()\n    elif name == "googlenet":\n        weights = models.GoogLeNet_Weights.IMAGENET1K_V1 if pretrained else None\n        network = models.googlenet(weights=weights, aux_logits=bool(pretrained),\n                                  transform_input=True, init_weights=False)\n        network.aux_logits, network.aux1, network.aux2 = False, None, None\n        width = network.fc.in_features\n        network.fc, network.dropout = nn.Identity(), nn.Identity()\n    elif name == "vgg16_bn":\n        original = models.vgg16_bn(weights=models.VGG16_BN_Weights.IMAGENET1K_V1 if pretrained else None)\n        # The 1000-class ImageNet classifier is not reused. The supplied CNN\n        # feature extractor is followed by our own compact sequential head.\n        network = nn.Sequential(original.features, nn.AdaptiveAvgPool2d((1, 1)), nn.Flatten())\n        width = 512\n        if cfg["freeze_early"]:\n            for block in original.features[:24]:\n                block.requires_grad_(False)\n    return nn.Sequential(network, nn.Sequential(nn.Dropout(cfg["dropout"]), nn.Linear(width, num_classes)))\n\n\ndef set_train(model):\n    model.train()\n    # Parameters and running mean/variance are frozen together in early ResNet blocks.\n    for module in model.modules():\n        if isinstance(module, nn.modules.batchnorm._BatchNorm) and not any(\n                parameter.requires_grad for parameter in module.parameters()):\n            module.eval()\n\n\ndef optimizer_for(model, cfg):\n    groups = []\n    for part, rate in ((model[0], cfg["lr_backbone"]), (model[1], cfg["lr_head"])):\n        for decay in (False, True):\n            params = [parameter for parameter in part.parameters() if parameter.requires_grad\n                      and (parameter.ndim > 1) == decay]\n            if params:\n                groups.append(dict(params=params, lr=rate, initial_lr=rate,\n                                   weight_decay=cfg["weight_decay"] if decay else 0.0))\n    return torch.optim.AdamW(groups)\n\n\nclass WarmEMA:\n    def __init__(self, model, decay):\n        self.model = copy.deepcopy(model).eval().requires_grad_(False)\n        self.decay, self.updates = float(decay), 0\n\n    @torch.no_grad()\n    def update(self, model):\n        self.updates += 1\n        decay = min(self.decay, (1 + self.updates) / (10 + self.updates))\n        current = model.state_dict()\n        for key, averaged in self.model.state_dict().items():\n            if averaged.is_floating_point():\n                averaged.lerp_(current[key], 1. - decay)\n            else:\n                averaged.copy_(current[key])\n\n\ndef _save_checkpoint(path, model, cfg, tables, job, epoch, fit_indices, holdout_indices, updates):\n    state = {key: value.detach().cpu().float().clone() if value.is_floating_point()\n             else value.detach().cpu().clone() for key, value in model.state_dict().items()}\n    checkpoint = dict(state_dict=state, config=cfg, classes=tables["classes"],\n                      num_classes=len(tables["classes"]), data_signature=tables["data_signature"],\n                      worker_protocol=PROTOCOL, job_signature=job_signature(job), phase=job["phase"],\n                      epoch=int(epoch), weight_kind="fixed_last_ema", ema_updates=updates,\n                      train_val_indices=fit_indices, holdout_val_indices=holdout_indices,\n                      source_signature=source_signature(),\n                      minimum_epochs=int(job.get("min_epochs", cfg["minimum_epochs"])),\n                      complete_protocol=(epoch >= int(job.get("min_epochs", cfg["minimum_epochs"]))\n                          if job["phase"] == "final" else epoch == cfg["epochs"]))\n    path = Path(path)\n    temporary = path.with_name(path.name + ".tmp")\n    torch.save(checkpoint, temporary)\n    os.replace(temporary, path)\n\n\ndef source_signature():\n    return {"geo_v7_45_worker.py": base._sha256(Path(__file__)),\n            "geo_worker.py": base._sha256(Path(base.__file__))}\n\n\ndef _training_result(job, cfg, tables, checkpoint, epochs, fitted, heldout, skipped, steps,\n                     started, stopped_reason=None, training_finished=False):\n    eligible = (epochs >= int(job.get("min_epochs", cfg["minimum_epochs"]))\n                if job["phase"] == "final" else epochs == cfg["epochs"])\n    exists = checkpoint.exists()\n    return dict(worker_protocol=PROTOCOL, complete_protocol=eligible, completed=eligible,\n        complete=eligible, eligible=eligible, training_finished=training_finished,\n        epochs_done=epochs, completed_epochs=epochs, epochs_requested=cfg["epochs"],\n        minimum_epochs=int(job.get("min_epochs", cfg["minimum_epochs"])),\n        stopped_early=bool(stopped_reason), stopped_reason=stopped_reason,\n        phase=job["phase"], config=cfg, job_signature=job_signature(job),\n        source_signature=source_signature(), data_signature=tables["data_signature"],\n        classes=tables["classes"], checkpoint=str(checkpoint) if exists else None,\n        checkpoint_sha256=base._sha256(checkpoint) if exists else None,\n        weight_kind="fixed_last_ema", last_epoch=epochs, train_images=len(tables["train"]),\n        fit_val_images=len(fitted), holdout_val_images=len(heldout),\n        training_rows_per_epoch=len(tables["train"]) + len(fitted) * cfg["val_repeat"],\n        train_val_indices=fitted, holdout_val_indices=heldout, heldout_evaluation=None,\n        training_performed=True, fit_test_images=0, skipped_steps=skipped, successful_steps=steps,\n        elapsed_seconds=time.time() - started)\n\n\n@torch.inference_mode()\ndef _evaluate(model, loader, device, job):\n    model.eval()\n    total, correct, loss = 0, 0, 0.\n    for images, labels in loader:\n        check_deadline(job)\n        images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)\n        with torch.autocast(device_type=device.type, enabled=device.type == "cuda"):\n            logits = model(images)\n        if not torch.isfinite(logits).all():\n            raise FloatingPointError("Nonfinite held-out logits")\n        loss += float(nn.functional.cross_entropy(logits.float(), labels, reduction="sum"))\n        correct += int(logits.argmax(1).eq(labels).sum())\n        total += len(labels)\n    return dict(count=total, correct=correct, accuracy=correct / total, loss=loss / total) if total else None\n\n\ndef train(job):\n    started, cfg = time.time(), dict(job["config"])\n    validate_config(cfg)\n    minimum = int(job.get("min_epochs", cfg["minimum_epochs"]))\n    if not 1 <= minimum <= cfg["epochs"]:\n        raise ValueError("Declared minimum must lie between one and the maximum epoch count")\n    check_deadline(job)\n    base.seed_everything(cfg["seed"])\n    device = base._device(job)\n    tables = load_metadata(job["data_dir"], include_test=False)\n    fit_indices, holdout_indices = validate_partition(job, len(tables["val"]))\n    output = Path(job["output_dir"])\n    output.mkdir(parents=True, exist_ok=True)\n    if any((output / name).exists() for name in ("best.pt", "checkpoint_a.pt", "checkpoint_b.pt", "history.json", "result.json")):\n        raise FileExistsError("Fits must start in a fresh attempt; cached/partial weights are never silently reused")\n    model = build_model(cfg, len(tables["classes"]), pretrained=True).to(device)\n    check_deadline(job)\n    optimizer = optimizer_for(model, cfg)\n    ema = WarmEMA(model, cfg["ema_decay"])\n    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")\n    transform = build_transform(cfg, training=True)\n    cache = job.get("cache_dir")\n    pieces = [ImageDataset(tables, job["data_dir"], "train", transform, cache_dir=cache)]\n    if fit_indices:\n        pieces.extend(ImageDataset(tables, job["data_dir"], "val", transform, fit_indices, cache)\n                      for _ in range(cfg["val_repeat"]))\n    dataset = ConcatDataset(pieces)\n    loader = base._loader(dataset, cfg, job, training=True)\n    if loader.drop_last:\n        raise ValueError("Every labeled training row must be visited; drop_last forbidden")\n    criterion = nn.CrossEntropyLoss(label_smoothing=cfg["label_smoothing"])\n    mixup = v2.MixUp(alpha=cfg["mixup_alpha"], num_classes=len(tables["classes"]))\n    cutmix = v2.CutMix(alpha=cfg["cutmix_alpha"], num_classes=len(tables["classes"]))\n    total_planned_steps = cfg["epochs"] * len(loader)\n    warmup_steps = max(1, cfg["warmup_epochs"] * len(loader))\n    history, skipped, steps, stopped = [], 0, 0, False\n    checkpoint = output / "checkpoint_a.pt"\n    expected = len(tables["train"]) + len(fit_indices) * cfg["val_repeat"]\n    for epoch in range(1, cfg["epochs"] + 1):\n        seen, loss_sum, correct, mixed = 0, 0., 0, 0\n        try:\n            check_deadline(job)\n            set_train(model)\n            for images, labels in loader:\n                check_deadline(job)\n                images = images.to(device, non_blocking=True)\n                labels = labels.to(device, non_blocking=True)\n                reference_labels = labels\n                if len(labels) > 1 and torch.rand((), device=device) < cfg["mix_p"]:\n                    images, labels = (mixup if torch.rand((), device=device) < .5 else cutmix)(images, labels)\n                    mixed += 1\n                factor = ((steps + 1) / warmup_steps if steps < warmup_steps else\n                          .5 * (1. + math.cos(math.pi * min(1., (steps - warmup_steps) /\n                                  max(1, total_planned_steps - warmup_steps)))))\n                for group in optimizer.param_groups:\n                    group["lr"] = group["initial_lr"] * factor\n                optimizer.zero_grad(set_to_none=True)\n                with torch.autocast(device_type=device.type, enabled=device.type == "cuda"):\n                    logits = model(images)\n                    loss = criterion(logits, labels)\n                if not torch.isfinite(loss) or not torch.isfinite(logits).all():\n                    raise FloatingPointError("Nonfinite training output")\n                scaler.scale(loss).backward()\n                scaler.unscale_(optimizer)\n                norm = nn.utils.clip_grad_norm_(model.parameters(), cfg["grad_clip"])\n                if device.type != "cuda" and not torch.isfinite(norm):\n                    raise FloatingPointError("Nonfinite CPU gradient")\n                old_scale = scaler.get_scale()\n                scaler.step(optimizer)\n                scaler.update()\n                successful = scaler.get_scale() >= old_scale\n                if successful:\n                    ema.update(model)\n                    steps += 1\n                else:\n                    skipped += 1\n                seen += len(reference_labels)\n                loss_sum += float(loss.detach()) * len(reference_labels)\n                correct += int(logits.detach().argmax(1).eq(reference_labels).sum())\n                check_deadline(job)\n            if seen != expected:\n                raise RuntimeError("Completed epoch did not visit every planned labeled row")\n        except BudgetExpired:\n            stopped = True\n            break\n        # Save only the last COMPLETED epoch. Interrupted batches must not enter\n        # either checkpoint or the accompanying result. No validation prediction\n        # is performed in this training function (screen/final both train only).\n        # The preceding result always points to the OTHER slot. An external\n        # deadline stop during saving therefore cannot corrupt its committed\n        # checkpoint/result pair. At most two full FP32 snapshots are retained.\n        checkpoint = output / ("checkpoint_a.pt" if epoch % 2 else "checkpoint_b.pt")\n        _save_checkpoint(checkpoint, ema.model, cfg, tables, job, epoch,\n                         fit_indices, holdout_indices, ema.updates)\n        row = dict(epoch=epoch, train_loss=loss_sum / seen,\n                   train_accuracy_against_pre_mix_labels=correct / seen, training_rows=seen,\n                   unique_train_images=len(tables["train"]), unique_fit_val_images=len(fit_indices),\n                   val_repeat=cfg["val_repeat"], mixed_batches=mixed, skipped_steps_total=skipped,\n                   successful_steps_total=steps, ema_updates=ema.updates,\n                   seconds=time.time() - started,\n                   learning_rates=[group["lr"] for group in optimizer.param_groups])\n        history.append(row)\n        base._json_write(output / "history.json", history)\n        snapshot = _training_result(job, cfg, tables, checkpoint, len(history),\n                                    fit_indices, holdout_indices, skipped, steps, started)\n        base._json_write(output / "result.json", snapshot)\n        print(json.dumps(dict(epoch=epoch, loss=row["train_loss"], seconds=row["seconds"])), flush=True)\n    result = _training_result(job, cfg, tables, checkpoint, len(history), fit_indices,\n                              holdout_indices, skipped, steps, started,\n                              stopped_reason="deadline" if stopped else None,\n                              training_finished=True)\n    base._json_write(output / "result.json", result)\n    return result\n\n\ndef d4_views(images, tta):\n    if tta == "none":\n        return [images]\n    if tta != "d4":\n        raise ValueError("Inference supports none or complete eight-view D4")\n    return [TF.rotate(value, angle) for value in (images, TF.hflip(images))\n            for angle in (0, 90, 180, 270)]\n\n\ndef _load_checkpoint(path):\n    return torch.load(path, map_location="cpu", weights_only=True)\n\n\ndef predict(job):\n    started, cfg = time.time(), dict(job["config"])\n    validate_config(cfg)\n    check_deadline(job)\n    if job.get("split") not in {"val", "test"}:\n        raise ValueError("Prediction supports held-out val or test only")\n    base.seed_everything(cfg["seed"])\n    device = base._device(job)\n    tables = load_metadata(job["data_dir"], include_test=job["split"] == "test")\n    digest = base._sha256(job["checkpoint"])\n    if job.get("checkpoint_sha256") and job["checkpoint_sha256"] != digest:\n        raise ValueError("Checkpoint hash differs")\n    checkpoint = _load_checkpoint(job["checkpoint"])\n    if checkpoint.get("worker_protocol") != PROTOCOL or not checkpoint.get("complete_protocol"):\n        raise ValueError("Only eligible full-epoch checkpoints may predict")\n    if checkpoint.get("source_signature") != source_signature():\n        raise ValueError("Checkpoint worker source differs; rerun the declared training code")\n    if checkpoint.get("config") != cfg or checkpoint.get("classes") != tables["classes"]:\n        raise ValueError("Checkpoint configuration/class order differs")\n    for key, value in checkpoint["data_signature"].items():\n        if tables["data_signature"].get(key) != value:\n            raise ValueError("Checkpoint labeled dataset signature differs")\n    split = job["split"]\n    indices = _indices(job.get("indices", list(range(len(tables[split])))), len(tables[split]), "indices")\n    if not indices:\n        raise ValueError("Prediction indices must not be empty")\n    if split == "val":\n        if checkpoint["phase"] != "screen" or set(indices) != set(checkpoint["holdout_val_indices"]):\n            raise ValueError("Validation prediction must use the train-only screening model\'s unseen val set")\n    model = build_model(cfg, len(tables["classes"]), pretrained=False).to(device)\n    model.load_state_dict(checkpoint["state_dict"], strict=True)\n    model.eval()\n    dataset = ImageDataset(tables, job["data_dir"], split, build_transform(cfg),\n                           indices, job.get("cache_dir"))\n    loader = base._loader(dataset, cfg, job)\n    probabilities = []\n    with torch.inference_mode():\n        for images, _labels in loader:\n            check_deadline(job)\n            images = images.to(device, non_blocking=True)\n            sum_probs = None\n            views = d4_views(images, job.get("tta", "none"))\n            for view in views:\n                check_deadline(job)\n                with torch.autocast(device_type=device.type, enabled=device.type == "cuda"):\n                    logits = model(view)\n                value = logits.float().softmax(dim=1)\n                if not torch.isfinite(value).all():\n                    raise FloatingPointError("Nonfinite prediction")\n                sum_probs = value if sum_probs is None else sum_probs + value\n            probabilities.append((sum_probs / len(views)).cpu().numpy())\n    probs = np.concatenate(probabilities).astype(np.float32)\n    if probs.shape != (len(indices), len(tables["classes"])) or not np.isfinite(probs).all():\n        raise ValueError("Prediction shape or values differ")\n    output = Path(job["output_dir"])\n    output.mkdir(parents=True, exist_ok=True)\n    if (output / "probs.npz").exists() or (output / "result.json").exists():\n        raise FileExistsError("Prediction attempt output already exists")\n    path = output / "probs.npz"\n    temporary = path.with_name(path.name + ".tmp")\n    frame = tables[split].iloc[indices]\n    with temporary.open("wb") as stream:\n        np.savez_compressed(stream, probs=probs, ids=np.asarray(frame["id"].tolist(), dtype=str),\n                            labels=frame["label"].to_numpy(dtype=np.int64) if split == "val" else\n                                   np.full(len(indices), -1, dtype=np.int64),\n                            classes=np.asarray(tables["classes"], dtype=str), indices=np.asarray(indices),\n                            class_labels=np.arange(len(tables["classes"]), dtype=np.int64),\n                            data_signature=json.dumps(tables["data_signature"], sort_keys=True),\n                            config_json=json.dumps(cfg, sort_keys=True), checkpoint_sha256=digest,\n                            job_signature=job_signature(job), worker_protocol=PROTOCOL)\n    os.replace(temporary, path)\n    result = dict(worker_protocol=PROTOCOL, complete_protocol=True, completed=True, complete=True,\n                  eligible=True, usable=True,\n                  phase="predict", config=cfg, split=split, indices=indices, tta=job.get("tta", "none"),\n                  rows=len(indices), forward_views=8 if job.get("tta") == "d4" else 1,\n                  probabilities=str(path), probs_path=str(path), probs_sha256=base._sha256(path),\n                  checkpoint_sha256=digest, data_signature=tables["data_signature"],\n                  source_signature=source_signature(),\n                  classes=tables["classes"], job_signature=job_signature(job),\n                  elapsed_seconds=time.time() - started, training_performed=False)\n    base._json_write(output / "result.json", result)\n    return result\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--job", required=True)\n    args = parser.parse_args()\n    job = json.loads(Path(args.job).read_text(encoding="utf-8"))\n    if job.get("mode") == "predict" or job.get("phase") == "predict":\n        result = predict(job)\n    else:\n        result = train(job)\n    print(json.dumps(result, ensure_ascii=False, allow_nan=False), flush=True)\n\n\nif __name__ == "__main__":\n    main()\n\n', 'geo_v7_45_search.py': '"""Team training controller: labeled selection, fresh <=45-minute final fit.\n\nOnly predict_final can access test. Screening is outside the final-fit timer.\nFinal checkpoints are FP32 last complete-epoch EMA, never best-on-val weights.\n"""\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\nimport signal\nimport subprocess\nimport sys\nimport tarfile\nimport time\nimport uuid\n\nimport numpy as np\nimport pandas as pd\nimport torch\nimport torchvision\n\nimport geo_v7_45_worker as worker\n\nPROTOCOL = "v7_final45_labeled_only_1"\nREPORT_NAME = "REPORT_V7_45_TO_SEND.txt"\nSUBMISSION_NAME = "submission_geo_v7_45.csv"\nARCHIVE_NAME = "geo_v7_45_recovery.tar.gz"\nCODE_FILES = ("geo_worker.py", "geo_search.py", "geo_v7_45_worker.py", "geo_v7_45_search.py", "geo_v7_45_baselines.py")\nMODEL_SPECS = (("r152", "resnet152", 2026), ("r101", "resnet101", 2028), ("inc", "inception_v3", 2029))\nTTAS = ("none", "d4")\nFINAL_LIMIT_SECONDS = 45 * 60\nTRAIN_STOP_SECONDS = 40 * 60\nMIN_FINAL_EPOCHS = 3\n\n\ndef write_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_name(path.name + ".tmp")\n    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")\n    os.replace(temporary, path)\n\n\ndef read_json(path):\n    return json.loads(Path(path).read_text(encoding="utf-8"))\n\n\ndef digest(path):\n    sha = hashlib.sha256()\n    with Path(path).open("rb") as stream:\n        for chunk in iter(lambda: stream.read(1024 * 1024), b""):\n            sha.update(chunk)\n    return sha.hexdigest()\n\n\ndef fingerprint(value):\n    return hashlib.sha256(json.dumps(value, sort_keys=True, separators=(",", ":"), allow_nan=False).encode()).hexdigest()\n\n\ndef probabilities(value, rows=None, classes=None):\n    value = np.asarray(value, dtype=np.float64)\n    if value.ndim != 2 or not value.size or (rows is not None and len(value) != rows):\n        raise ValueError("Invalid prediction row count")\n    if classes is not None and value.shape[1] != classes:\n        raise ValueError("Invalid prediction class count")\n    if not np.isfinite(value).all() or (value < -1e-7).any() or (value > 1.000001).any():\n        raise ValueError("Invalid probabilities")\n    if not np.allclose(value.sum(1), 1, atol=2e-4, rtol=0):\n        raise ValueError("Probability rows do not sum to one")\n    return value\n\n\ndef mixture(values, weights):\n    if not weights or len(weights) > 2 or any(v <= 0 for v in weights.values()) or abs(sum(weights.values()) - 1) > 1e-9:\n        raise ValueError("Recipe must contain one or two positive contributors summing to one")\n    shapes = {np.asarray(values[k]).shape for k in weights}\n    if len(shapes) != 1:\n        raise ValueError("Contributor shapes differ")\n    return probabilities(sum(probabilities(values[k]) * v for k, v in weights.items()))\n\n\ndef score(value, labels):\n    labels = np.asarray(labels, dtype=np.int64)\n    value = probabilities(value, len(labels))\n    if labels.ndim != 1 or (labels < 0).any() or (labels >= value.shape[1]).any():\n        raise ValueError("Invalid validation labels")\n    correct = int((value.argmax(1) == labels).sum())\n    return dict(correct=correct, count=len(labels), accuracy=correct / len(labels),\n                loss=float(-np.log(np.clip(value[np.arange(len(labels)), labels], 1e-12, 1)).mean()))\n\n\ndef recipes():\n    return [("r152", {"r152": 1.}), ("r101", {"r101": 1.}), ("inc", {"inc": 1.}),\n            ("r152_inc_60_40", {"r152": .6, "inc": .4}),\n            ("r152_inc_40_60", {"r152": .4, "inc": .6}),\n            ("r152_r101_70_30", {"r152": .7, "r101": .3})]\n\n\ndef choose_recipe(predictions, labels):\n    rows = []\n    for tta in TTAS:\n        for name, weights in recipes():\n            rows.append(dict(score(mixture(predictions[tta], weights), labels), name=name + "_" + tta,\n                             weights=weights, tta=tta, views=1 if tta == "none" else 8))\n    rows.sort(key=lambda row: (-row["correct"], row["loss"], row["views"], row["name"]))\n    return rows[0], rows\n\n\ndef stop_process(process):\n    """Bounded termination of worker and Linux DataLoader process group."""\n    try:\n        if os.name == "posix":\n            os.killpg(process.pid, signal.SIGTERM)\n        elif process.poll() is None:\n            process.terminate()\n        process.wait(timeout=4)\n    except ProcessLookupError:\n        return\n    except subprocess.TimeoutExpired:\n        if os.name == "posix":\n            os.killpg(process.pid, signal.SIGKILL)\n        else:\n            process.kill()\n        process.wait(timeout=4)\n\n\nclass Pool:\n    def __init__(self, data_dir, run_dir, code_dir, cache_dir, workers=2):\n        self.data_dir, self.output_dir, self.code_dir, self.cache_dir = map(\n            lambda p: Path(p).resolve(), (data_dir, run_dir, code_dir, cache_dir))\n        for other in (self.data_dir, self.cache_dir):\n            if other.is_relative_to(self.output_dir) or self.output_dir.is_relative_to(other):\n                raise ValueError("Run output must be separate from data and image cache")\n        self.output_dir.mkdir(parents=True, exist_ok=True)\n        self.workers, self.started = int(workers), time.time()\n        # No test table or image is read during initialization or selection.\n        metadata = worker.load_metadata(self.data_dir, include_test=False)\n        self.train, self.val = metadata["train"], metadata["val"]\n        self.classes = list(metadata["classes"])\n        self.class_labels = list(range(len(self.classes)))\n        self.data_signature = {name: digest(self.data_dir / name) for name in ("classes.csv", "train.csv", "val.csv")}\n        self.source_hashes = {name: digest(self.code_dir / name) for name in CODE_FILES}\n        self.configs = {key: worker.config_for_model(model, seed, trial_id="v7_45_" + key)\n                        for key, model, seed in MODEL_SPECS}\n        self.gpus = list(range(torch.cuda.device_count()))[:2]\n        if not self.gpus:\n            raise RuntimeError("Enable Kaggle GPU; CPU final training is not supported")\n        self.failures, self.events = [], []\n        self.environment = dict(python=sys.version, torch=torch.__version__, torchvision=torchvision.__version__,\n                                numpy=np.__version__, pandas=pd.__version__, cuda=torch.version.cuda,\n                                cudnn=torch.backends.cudnn.version(),\n                                gpu_names=[torch.cuda.get_device_name(gpu) for gpu in self.gpus])\n        context = dict(protocol=PROTOCOL, data_signature=self.data_signature, source_hashes=self.source_hashes,\n                       configs=self.configs, recipes=recipes(), tta=TTAS, final_limit_seconds=FINAL_LIMIT_SECONDS,\n                       training_stop_seconds=TRAIN_STOP_SECONDS, final_reuse=False, no_test_fitting=True)\n        context = json.loads(json.dumps(context))\n        prior = self.output_dir / "run_context.json"\n        if prior.exists() and read_json(prior) != context:\n            raise RuntimeError("Existing RUN_DIR has different source/data/config. Use a new RUN_DIR.")\n        write_json(prior, context)\n        write_json(self.output_dir / "environment.json", self.environment)\n\n    def event(self, kind, **details):\n        row = dict(time_unix=time.time(), kind=kind, **details)\n        self.events.append(row)\n        write_json(self.output_dir / "events.json", self.events)\n        print(f"[V7 final45] {kind}: {details}", flush=True)\n\n    def identity(self, payload):\n        keep = {k: v for k, v in payload.items() if k not in {"deadline_unix", "output_dir"}}\n        keep.update(controller_protocol=PROTOCOL, source_hashes=self.source_hashes, data_signature=self.data_signature,\n                    data_dir=str(self.data_dir), cache_dir=str(self.cache_dir), workers=self.workers)\n        return json.loads(json.dumps(keep))\n\n    def completed(self, directory, identity, prediction=False):\n        try:\n            if read_json(directory / "controller_identity.json") != identity:\n                return None\n            result = read_json(directory / "result.json")\n            if result.get("config") != identity["config"] or not result.get("complete_protocol") or result.get("classes") != self.classes:\n                return None\n            if result.get("worker_protocol") != worker.PROTOCOL:\n                return None\n            job = read_json(directory / "job.json")\n            if result.get("job_signature") != worker.job_signature(job) or result.get("phase") != identity["phase"]:\n                return None\n            if any(result.get("data_signature", {}).get(k) != v for k, v in self.data_signature.items()):\n                return None\n            expected_source = {k: self.source_hashes[k] for k in ("geo_worker.py", "geo_v7_45_worker.py")}\n            if result.get("source_signature") != expected_source:\n                return None\n            if prediction:\n                checkpoint = Path(identity["checkpoint"])\n                prediction_file = directory / "probs.npz"\n                if not prediction_file.is_file() or result.get("checkpoint_sha256") != digest(checkpoint):\n                    return None\n                if result.get("probs_sha256") != digest(prediction_file):\n                    return None\n            else:\n                epochs = int(result.get("completed_epochs", result.get("epochs_done", 0)))\n                if not result.get("eligible") or epochs < (MIN_FINAL_EPOCHS if identity["phase"] == "final" else int(identity["config"]["epochs"])):\n                    return None\n                if epochs > int(identity["config"]["epochs"]) or result.get("weight_kind") != "fixed_last_ema":\n                    return None\n                if result.get("train_images") != len(self.train) or result.get("fit_test_images") != 0:\n                    return None\n                if result.get("fit_val_images") != (len(self.val) if identity["phase"] == "final" else 0):\n                    return None\n                checkpoint = Path(result.get("checkpoint", directory / "best.pt")).resolve()\n                if not checkpoint.is_relative_to(directory.resolve()):\n                    return None\n                if not checkpoint.is_file() or result.get("checkpoint_sha256") != digest(checkpoint):\n                    return None\n            return dict(result, key=identity["key"], phase=identity["phase"], config=identity["config"],\n                        result_dir=str(directory), checkpoint=str(checkpoint), checkpoint_sha256=digest(checkpoint),\n                        controller_identity=fingerprint(identity))\n        except (OSError, ValueError, KeyError, TypeError):\n            return None\n\n    def run(self, payloads, prediction=False, deadline=None, reuse=True, attempt_root=None):\n        queue, done, active = [], [], []\n        for payload in payloads:\n            identity = self.identity(payload)\n            name = payload["key"] + ("_" + payload["tta"] if prediction else "")\n            parent = (Path(attempt_root) if attempt_root else self.output_dir / "screen") / ("predict" if prediction else "train") / name\n            parent.mkdir(parents=True, exist_ok=True)\n            previous = None\n            if reuse:\n                for directory in sorted(parent.glob("attempt_*")):\n                    previous = self.completed(directory, identity, prediction)\n                    if previous:\n                        break\n            if previous:\n                done.append(previous)\n                self.event("reuse_complete_screen", key=name)\n            else:\n                directory = parent / ("attempt_" + uuid.uuid4().hex[:12])\n                queue.append((payload, identity, directory, name))\n        try:\n            while queue or active:\n                now = time.time()\n                if deadline is not None and now >= deadline:\n                    for item in list(active):\n                        running = item["process"].poll() is None\n                        if running:\n                            stop_process(item["process"])\n                        item["log"].close()\n                        result = self.completed(item["directory"], item["identity"], prediction) if running or item["process"].returncode == 0 else None\n                        if result:\n                            done.append(result)\n                        else:\n                            self.failures.append(dict(key=item["name"], reason="deadline_no_eligible_result"))\n                    active.clear()\n                    self.failures.extend(dict(key=name, reason="deadline_before_launch") for _, _, _, name in queue)\n                    queue.clear()\n                    break\n                free = [gpu for gpu in self.gpus if gpu not in [item["gpu"] for item in active]]\n                while free and queue:\n                    payload, identity, directory, name = queue.pop(0)\n                    gpu = free.pop(0)\n                    directory.mkdir(parents=True)\n                    write_json(directory / "controller_identity.json", identity)\n                    job_deadline = deadline if deadline is not None else time.time() + (2 if prediction else 4) * 3600\n                    job = dict(payload, data_dir=str(self.data_dir), output_dir=str(directory), cache_dir=str(self.cache_dir),\n                               workers=self.workers, deadline_unix=job_deadline, allow_cpu=False)\n                    write_json(directory / "job.json", job)\n                    log = (directory / "worker.log").open("w", encoding="utf-8")\n                    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), PYTHONUNBUFFERED="1", OMP_NUM_THREADS="2",\n                               MKL_NUM_THREADS="2", CUBLAS_WORKSPACE_CONFIG=":4096:8")\n                    try:\n                        process = subprocess.Popen([sys.executable, str(self.code_dir / "geo_v7_45_worker.py"),\n                                                    "--job", str(directory / "job.json")],\n                                                   stdout=log, stderr=subprocess.STDOUT, cwd=str(self.code_dir), env=env,\n                                                   start_new_session=os.name == "posix")\n                    except BaseException:\n                        log.close()\n                        raise\n                    active.append(dict(process=process, log=log, directory=directory, identity=identity,\n                                       name=name, gpu=gpu, deadline=job_deadline))\n                    self.event("launch_prediction" if prediction else "launch_training", key=name, gpu=gpu)\n                for item in list(active):\n                    process = item["process"]\n                    if process.poll() is None and time.time() < item["deadline"]:\n                        continue\n                    stopped = process.poll() is None\n                    if stopped:\n                        stop_process(process)\n                    item["log"].close()\n                    result = self.completed(item["directory"], item["identity"], prediction) if stopped or process.returncode == 0 else None\n                    if result:\n                        done.append(result)\n                        self.event("worker_complete", key=item["name"], epochs=result.get("completed_epochs"))\n                    else:\n                        self.failures.append(dict(key=item["name"], reason="no_eligible_result", returncode=process.returncode,\n                                                  timed_out=stopped, result_dir=str(item["directory"])))\n                    active.remove(item)\n                if active:\n                    time.sleep(.25)\n        finally:\n            for item in active:\n                stop_process(item["process"])\n                item["log"].close()\n            write_json(self.output_dir / "failures.json", self.failures)\n        return done\n\n\ndef prediction_payload(record, split, tta):\n    return dict(mode="predict", phase="predict", key=record["key"], config=record["config"],\n                checkpoint=record["checkpoint"], checkpoint_sha256=digest(record["checkpoint"]), split=split, tta=tta)\n\n\ndef read_prediction(record, ids, classes):\n    with np.load(Path(record["result_dir"]) / "probs.npz", allow_pickle=False) as archive:\n        got_ids = archive["ids"].astype(str).tolist()\n        got_classes = archive["classes"].astype(str).tolist()\n        if got_ids != [str(x) for x in ids] or got_classes != list(classes):\n            raise ValueError("ID/class order differs from the frozen prediction protocol")\n        value = probabilities(archive["probs"], len(ids), len(classes))\n        if "checkpoint_sha256" not in archive or str(archive["checkpoint_sha256"].item()) != record["checkpoint_sha256"]:\n            raise ValueError("Prediction refers to different checkpoint bytes")\n        return dict(probs=value, labels=archive["labels"].astype(np.int64).tolist())\n\n\ndef run_screen(pool):\n    pool.event("cache_labeled_only_before_screen")\n    worker.build_cache(pool.data_dir, pool.cache_dir, side=320, include_test=False)\n    jobs = [dict(mode="train", phase="screen", key=key, config=config,\n                 train_val_indices=[], holdout_val_indices=list(range(len(pool.val))))\n            for key, config in pool.configs.items()]\n    fits = pool.run(jobs)\n    if len(fits) != len(jobs) or {x["key"] for x in fits} != set(pool.configs):\n        raise RuntimeError("All three complete train-only screening fits are required; no test accessed")\n    write_json(pool.output_dir / "screen_fits.json", fits)\n    predictions = pool.run([prediction_payload(record, "val", tta) for record in fits for tta in TTAS], prediction=True)\n    if len(predictions) != len(fits) * len(TTAS):\n        raise RuntimeError("Validation inference incomplete; no recipe frozen and no test accessed")\n    write_json(pool.output_dir / "screen_predictions.json", predictions)\n    return fits\n\n\ndef select_recipe(pool):\n    fits = read_json(pool.output_dir / "screen_fits.json")\n    predictions = read_json(pool.output_dir / "screen_predictions.json")\n    values = {tta: {} for tta in TTAS}\n    individual = []\n    for record in predictions:\n        job = read_json(Path(record["result_dir"]) / "job.json")\n        array = read_prediction(record, pool.val["id"].tolist(), pool.classes)\n        if array["labels"] != pool.val["label"].tolist():\n            raise ValueError("Validation labels and prediction rows are not aligned")\n        values[job["tta"]][record["key"]] = array["probs"]\n        individual.append(dict(key=record["key"], tta=job["tta"], **score(array["probs"], array["labels"])))\n    selected, table = choose_recipe(values, pool.val["label"].to_numpy())\n    recipe = dict(protocol=PROTOCOL, selected=selected, configs={key: pool.configs[key] for key in selected["weights"]},\n                  data_signature=pool.data_signature, source_hashes=pool.source_hashes,\n                  screen_checkpoint_hashes={row["key"]: row["checkpoint_sha256"] for row in fits},\n                  final_policy=dict(max_epochs=14, minimum_epochs=MIN_FINAL_EPOCHS, stop_seconds=TRAIN_STOP_SECONDS,\n                                    limit_seconds=FINAL_LIMIT_SECONDS, weight_kind="last complete epoch FP32 EMA",\n                                    final_weight_reuse=False, val_repeat=2),\n                  no_test_fitting=True, final_validation_accuracy_available=False,\n                  validation_note="Train-only held-out val; reused 12-recipe selection, not a new independent test.")\n    recipe["recipe_fingerprint"] = fingerprint(recipe)\n    write_json(pool.output_dir / "validation_recipes.json", table)\n    write_json(pool.output_dir / "screen_model_metrics.json", individual)\n    write_json(pool.output_dir / "selected_recipe.json", recipe)\n    pool.event("recipe_frozen_before_test", name=selected["name"], accuracy=selected["accuracy"])\n    return recipe\n\n\ndef verify_frozen_recipe(pool):\n    recipe = read_json(pool.output_dir / "selected_recipe.json")\n    signature = recipe.pop("recipe_fingerprint")\n    if fingerprint(recipe) != signature or recipe.get("source_hashes") != pool.source_hashes or recipe.get("data_signature") != pool.data_signature:\n        raise RuntimeError("Frozen selection recipe was changed; rerun labeled screening")\n    recipe["recipe_fingerprint"] = signature\n    return recipe\n\n\ndef run_final_train(pool):\n    recipe = verify_frozen_recipe(pool)\n    selected = recipe["selected"]\n    attempt = pool.output_dir / "final_attempts" / (time.strftime("%Y%m%d_%H%M%S") + "_" + uuid.uuid4().hex[:8])\n    attempt.mkdir(parents=True)\n    # Preserve old artifacts recoverably, but remove their misleading top-level\n    # links before a fresh attempt so a failed run cannot look like a new CSV.\n    previous = attempt / "previous_output"\n    for name in (SUBMISSION_NAME, "submission_summary.json", "final_ensemble_probabilities.npz", "final_fits.json",\n                 "frozen_final.json", "reproduce_final_recipe.json"):\n        path = pool.output_dir / name\n        if path.is_file():\n            previous.mkdir(exist_ok=True)\n            os.replace(path, previous / name)\n    # Every call trains fresh weights. Neither completed screen nor prior final\n    # results can replace this timed final fit on train+val.\n    started_wall, started_monotonic = time.time(), time.monotonic()\n    timer = dict(attempt=str(attempt), start_time=started_wall, end_time=None, elapsed_seconds=None,\n                 elapsed_minutes=None, within_limit=False, training_performed=True, final_weight_reuse=False,\n                 final_limit_seconds=FINAL_LIMIT_SECONDS, training_stop_seconds=TRAIN_STOP_SECONDS,\n                 included="model initialization, process startup, all final training, EMA checkpoint saves and process teardown",\n                 excluded="screening, validation model comparison and test prediction", status="running")\n    write_json(pool.output_dir / "final_training_time.json", timer)\n    write_json(attempt / "frozen_selection.json", recipe)\n    fits, error = [], None\n    try:\n        jobs = [dict(mode="train", phase="final", key=key, config=recipe["configs"][key], min_epochs=MIN_FINAL_EPOCHS,\n                     train_val_indices=list(range(len(pool.val))), holdout_val_indices=[])\n                for key in selected["weights"]]\n        fits = pool.run(jobs, deadline=started_wall + TRAIN_STOP_SECONDS, reuse=False, attempt_root=attempt)\n        if len(fits) != len(jobs) or {row["key"] for row in fits} != set(selected["weights"]):\n            raise RuntimeError("Not all selected final models reached three complete epochs; no partial/reweighted CSV is allowed")\n    except BaseException as exc:\n        error = exc\n    finally:\n        elapsed = time.monotonic() - started_monotonic\n        timer.update(end_time=time.time(), elapsed_seconds=elapsed, elapsed_minutes=elapsed / 60,\n                     within_limit=elapsed <= FINAL_LIMIT_SECONDS,\n                     status="complete" if error is None and elapsed <= FINAL_LIMIT_SECONDS else "failed")\n        write_json(pool.output_dir / "final_training_time.json", timer)\n        write_json(attempt / "final_training_time.json", timer)\n        print(f"Elapsed time: {timer[\'elapsed_minutes\']:.2f} minutes (final training only)", flush=True)\n    if error is not None:\n        raise error\n    if not timer["within_limit"]:\n        raise RuntimeError("Final training exceeded 45 minutes. No CSV can be exported from this attempt.")\n    frozen = dict(protocol=PROTOCOL, selection_fingerprint=recipe["recipe_fingerprint"], selected=selected,\n                  final_training_time=timer, models=[dict(key=row["key"], config=row["config"], checkpoint=row["checkpoint"],\n                    checkpoint_sha256=row["checkpoint_sha256"], completed_epochs=row.get("completed_epochs", row.get("epochs_done")),\n                    weight_kind=row.get("weight_kind"), result_dir=row["result_dir"]) for row in fits],\n                  actual_recipe_matches_validation_selected_weights=True, final_validation_accuracy=None)\n    frozen["final_fingerprint"] = fingerprint(frozen)\n    write_json(pool.output_dir / "final_fits.json", fits)\n    write_json(pool.output_dir / "frozen_final.json", frozen)\n    write_json(attempt / "frozen_final.json", frozen)\n    reproduce = dict(protocol=PROTOCOL, actual_completed_epochs={row["key"]: row.get("completed_epochs", row.get("epochs_done")) for row in fits},\n                     configs={row["key"]: row["config"] for row in fits}, selected=selected,\n                     data_signature=pool.data_signature, source_hashes=pool.source_hashes, environment=pool.environment,\n                     final_fingerprint=frozen["final_fingerprint"], final_limit_seconds=FINAL_LIMIT_SECONDS,\n                     exact_inference_note="Use these archived FP32 checkpoint bytes for the exact submitted CSV; seeds alone do not promise bit-exact retraining.",\n                     fixed_epoch_retraining_note="A reviewer can stop each refit after actual_completed_epochs rather than hardware-dependent stopping, while preserving the original planned 14-epoch learning-rate schedule and all configs. Do NOT change cfg.epochs to the shorter count: that changes the schedule. Include those fits in the same 45-minute timer.")\n    write_json(pool.output_dir / "reproduce_final_recipe.json", reproduce)\n    write_json(attempt / "reproduce_final_recipe.json", reproduce)\n    pool.event("all_final_weights_frozen", elapsed_minutes=timer["elapsed_minutes"], actual_epochs={r["key"]: r["completed_epochs"] for r in fits})\n    return fits\n\n\ndef verify_final(pool):\n    recipe = verify_frozen_recipe(pool)\n    final = read_json(pool.output_dir / "frozen_final.json")\n    signature = final.pop("final_fingerprint")\n    if fingerprint(final) != signature:\n        raise RuntimeError("Frozen final model manifest was changed")\n    final["final_fingerprint"] = signature\n    timer = read_json(pool.output_dir / "final_training_time.json")\n    if timer != final["final_training_time"] or timer.get("status") != "complete" or not timer.get("training_performed"):\n        raise RuntimeError("Current final attempt did not complete; an older final manifest cannot be reused")\n    if not timer.get("within_limit") or not 0 < float(timer["elapsed_seconds"]) <= FINAL_LIMIT_SECONDS:\n        raise RuntimeError("Final training timing is not valid for the 45-minute requirement")\n    if final["selection_fingerprint"] != recipe["recipe_fingerprint"] or final["selected"] != recipe["selected"]:\n        raise RuntimeError("Final models differ from validation-frozen selection")\n    if len(final["models"]) != len(recipe["selected"]["weights"]) or {row["key"] for row in final["models"]} != set(recipe["selected"]["weights"]):\n        raise RuntimeError("Final contributor set differs; renormalizing a partial ensemble is forbidden")\n    for row in final["models"]:\n        if int(row["completed_epochs"]) < MIN_FINAL_EPOCHS or digest(row["checkpoint"]) != row["checkpoint_sha256"]:\n            raise RuntimeError("Final checkpoint identity or complete-epoch count changed")\n    return recipe, final\n\n\ndef test_metadata(pool):\n    """Prediction-only table access; no test stats, labels, balance or fitting."""\n    table = pd.read_csv(pool.data_dir / "test.csv", dtype={"id": str}, keep_default_na=False)\n    if "id" not in table or not len(table) or table["id"].duplicated().any():\n        raise ValueError("test.csv needs each nonempty image id exactly once")\n    ids = table["id"].tolist()\n    if any(not item.strip() for item in ids):\n        raise ValueError("Empty test image ID")\n    template = pool.data_dir / "sample_submission_example.csv"\n    if template.exists():\n        sample = pd.read_csv(template, dtype=str, keep_default_na=False)\n        if len(sample.columns) != 2 or sample.columns[0] != "id" or sample.columns[1] not in {"label", "target"}:\n            raise ValueError("Sample submission columns must be id,label or id,target")\n        order, target = sample["id"].tolist(), str(sample.columns[1])\n        if len(order) != len(ids) or len(set(order)) != len(order) or set(order) != set(ids):\n            raise ValueError("Submission template IDs do not equal test IDs")\n    else:\n        order, target = ids, "label"\n    return ids, order, target\n\n\ndef predict_final(pool):\n    # All guard checks occur BEFORE test.csv, test image cache or test pixels.\n    recipe, frozen = verify_final(pool)\n    ids, order, target = test_metadata(pool)\n    pool.event("test_prediction_only_after_all_final_weights_frozen")\n    worker.build_cache(pool.data_dir, pool.cache_dir, side=320, include_test=True)\n    jobs = [prediction_payload(record, "test", frozen["selected"]["tta"]) for record in frozen["models"]]\n    root = Path(frozen["final_training_time"]["attempt"]) / "test_inference"\n    records = pool.run(jobs, prediction=True, reuse=False, attempt_root=root)\n    if len(records) != len(jobs) or {row["key"] for row in records} != set(frozen["selected"]["weights"]):\n        raise RuntimeError("Incomplete final test inference; no partial submission is generated")\n    values = {row["key"]: read_prediction(row, ids, pool.classes)["probs"] for row in records}\n    value = mixture(values, frozen["selected"]["weights"])\n    predicted = value.argmax(1)\n    position = {item: index for index, item in enumerate(ids)}\n    frame = pd.DataFrame({"id": order, target: [int(predicted[position[item]]) for item in order]})\n    destination = pool.output_dir / SUBMISSION_NAME\n    temporary = destination.with_name(destination.name + ".tmp")\n    frame.to_csv(temporary, index=False)\n    os.replace(temporary, destination)\n    np.savez_compressed(pool.output_dir / "final_ensemble_probabilities.npz", ids=np.asarray(ids, dtype=str),\n                        probs=value.astype(np.float32), classes=np.asarray(pool.classes, dtype=str))\n    summary = dict(filename=SUBMISSION_NAME, sha256=digest(destination), rows=len(frame), target_column=target,\n                   selection_fingerprint=recipe["recipe_fingerprint"], final_fingerprint=frozen["final_fingerprint"],\n                   weights=frozen["selected"]["weights"], tta=frozen["selected"]["tta"],\n                   actual_epochs={row["key"]: row["completed_epochs"] for row in frozen["models"]},\n                   checkpoint_sha256={row["key"]: row["checkpoint_sha256"] for row in frozen["models"]},\n                   final_training_minutes=frozen["final_training_time"]["elapsed_minutes"], test_only_predicted=True,\n                   no_test_class_balance_adjustment=True, no_test_pseudo_labels=True, final_validation_accuracy=None)\n    write_json(pool.output_dir / "submission_summary.json", summary)\n    write_json(root / "submission_summary.json", summary)\n    pool.event("final_csv_complete", rows=len(frame), final_training_minutes=summary["final_training_minutes"])\n    return summary\n\n\ndef make_report(pool, archive_status=None):\n    def optional(name, default=None):\n        path = pool.output_dir / name\n        return read_json(path) if path.is_file() else default\n    timer = optional("final_training_time.json")\n    submission = optional("submission_summary.json")\n    final = optional("frozen_final.json")\n    # A failed new timed attempt must not be reported as an older valid CSV.\n    current = bool(timer and final and timer == final.get("final_training_time") and timer.get("status") == "complete")\n    if not current or (submission and final and submission.get("final_fingerprint") != final.get("final_fingerprint")):\n        submission = None\n    baseline = optional("baselines/results.json")\n    report = dict(protocol=PROTOCOL, all_test_labels_unknown=True, public_score=None,\n                  train_and_val_used_for_final_training=True, test_only_used_for_prediction=True,\n                  no_test_stats_or_fitting=True, no_test_pseudo_labels=True, no_test_class_balance_adjustment=True,\n                  selection=optional("selected_recipe.json"), screen_model_metrics=optional("screen_model_metrics.json", []),\n                  validation_recipes=optional("validation_recipes.json", []), baseline_comparison=baseline,\n                  validation_note="Train-only fits evaluated on held-out supplied val; reused recipe selection is not an independent test.",\n                  final_validation_accuracy=None, final_training_time=timer, final_attempt_current=current,\n                  actual_final_models=[{k: row.get(k) for k in ("key", "config", "completed_epochs", "checkpoint_sha256", "weight_kind")}\n                                       for row in final.get("models", [])] if current else [],\n                  submission=submission, failures=pool.failures, environment=pool.environment,\n                  total_run_minutes=(time.time() - pool.started) / 60, archive=archive_status,\n                  timing_note="45-minute limit applies only to fresh final training, including initialization/save/process teardown, not screening or test prediction.")\n    write_json(pool.output_dir / "report_v7_45.json", report)\n    text = "GEO V7 FINAL45: AGGREGATE REPORT\\n\\nTrain-only screening held val out; final training used permitted train+val.\\nTest only predicted after recipe and all final weights were frozen.\\nNo final-model validation accuracy is claimed. No test fitting, pseudolabels or class-prior correction.\\n\\n"\n    (pool.output_dir / REPORT_NAME).write_text(text + json.dumps(report, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")\n    return report\n\n\ndef bundle(pool):\n    """Private complete replay package; original images/uint8 caches excluded."""\n    root = pool.output_dir.resolve()\n    files = []\n    for path in sorted(root.rglob("*")):\n        if path.is_file() and not path.is_symlink() and path.suffix.lower() in {".pt", ".json", ".csv", ".npz", ".txt", ".log", ".py"}:\n            if path.name != "recovery_manifest.json" and not path.resolve().is_relative_to(pool.data_dir) and not path.resolve().is_relative_to(pool.cache_dir):\n                files.append(("run/" + path.relative_to(root).as_posix(), path))\n    files += [("code/" + name, pool.code_dir / name) for name in CODE_FILES]\n    manifest = dict(protocol=PROTOCOL, files={name: dict(sha256=digest(path), size=path.stat().st_size) for name, path in files},\n                    original_images_included=False, image_caches_included=False, includes_fp32_training_checkpoints=True)\n    write_json(root / "recovery_manifest.json", manifest)\n    destination = root / ARCHIVE_NAME\n    temporary = destination.with_name(destination.name + ".tmp")\n    with tarfile.open(temporary, "w:gz", compresslevel=1) as archive:\n        archive.add(root / "recovery_manifest.json", arcname="manifest.json", recursive=False)\n        for name, path in files:\n            archive.add(path, arcname=name, recursive=False)\n    os.replace(temporary, destination)\n    result = dict(created=True, filename=ARCHIVE_NAME, size_bytes=destination.stat().st_size, sha256=digest(destination),\n                  files=len(files), no_original_images=True, no_image_caches=True)\n    write_json(root / "archive_status.json", result)\n    return result\n', 'geo_v7_45_baselines.py': '"""Train-only required comparisons; explicitly outside final-training timer."""\nfrom __future__ import annotations\n\nimport gc\nimport time\nfrom pathlib import Path\n\nimport torch\nfrom torch import nn\nfrom torchvision.transforms import v2\n\nimport geo_worker as base\nimport geo_v7_45_worker as worker\n\n\ndef baseline_model(name, classes):\n    if name == "mlp":\n        return nn.Sequential(nn.Flatten(), nn.Linear(3 * 64 * 64, 512), nn.ReLU(),\n                             nn.Dropout(.3), nn.Linear(512, 128), nn.ReLU(), nn.Linear(128, classes))\n    if name != "cnn":\n        raise ValueError("Baseline must be mlp or cnn")\n    layers = []\n    previous = 3\n    for width in (16, 32, 64, 128):\n        layers.extend([nn.Conv2d(previous, width, 3, padding=1, bias=False), nn.BatchNorm2d(width),\n                       nn.ReLU(), nn.Conv2d(width, width, 3, padding=1, bias=False),\n                       nn.BatchNorm2d(width), nn.ReLU(), nn.MaxPool2d(2)])\n        previous = width\n    return nn.Sequential(*layers, nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(.3), nn.Linear(128, classes))\n\n\ndef transform(training):\n    ops = [v2.ToImage(), v2.Resize((64, 64), antialias=True)]\n    if training:\n        ops += [v2.RandomHorizontalFlip(), v2.RandomVerticalFlip(),\n                v2.RandomChoice([v2.RandomRotation((angle, angle)) for angle in (0, 90, 180, 270)])]\n    return v2.Compose(ops + [v2.ToDtype(torch.float32, scale=True), v2.Normalize(base.MEAN, base.STD)])\n\n\ndef run_baselines(data_dir, output_dir, cache_dir, epochs=8, workers=2, allow_cpu=False):\n    if not isinstance(epochs, int) or epochs < 1:\n        raise ValueError("Positive baseline epochs required")\n    device = base._device(dict(allow_cpu=allow_cpu))\n    output = Path(output_dir) / "baselines"\n    output.mkdir(parents=True, exist_ok=True)\n    tables = worker.load_metadata(data_dir, include_test=False)\n    rows = []\n    for name in ("mlp", "cnn"):\n        started = time.time()\n        base.seed_everything(2026)  # Before initialization, including isolated reruns.\n        model = baseline_model(name, len(tables["classes"])).to(device)\n        cfg = dict(seed=2026, batch_size=128)\n        job = dict(workers=workers)\n        train = base._loader(worker.ImageDataset(tables, data_dir, "train", transform(True), cache_dir=cache_dir),\n                             cfg, job, training=True)\n        val = base._loader(worker.ImageDataset(tables, data_dir, "val", transform(False), cache_dir=cache_dir), cfg, job)\n        optimizer = torch.optim.AdamW(model.parameters(), lr=.001, weight_decay=.0001)\n        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, epochs)\n        scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")\n        history = []\n        for epoch in range(1, epochs + 1):\n            model.train()\n            count, correct, loss_total = 0, 0, 0.\n            for images, labels in train:\n                images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)\n                optimizer.zero_grad(set_to_none=True)\n                with torch.autocast(device_type=device.type, enabled=device.type == "cuda"):\n                    logits = model(images)\n                    loss = nn.functional.cross_entropy(logits, labels)\n                if not torch.isfinite(loss):\n                    raise FloatingPointError("Nonfinite baseline loss")\n                scaler.scale(loss).backward()\n                scaler.unscale_(optimizer)\n                nn.utils.clip_grad_norm_(model.parameters(), 2.)\n                scaler.step(optimizer)\n                scaler.update()\n                count += len(labels)\n                correct += int(logits.detach().argmax(1).eq(labels).sum())\n                loss_total += float(loss.detach()) * len(labels)\n            if count != len(tables["train"]):\n                raise RuntimeError("Baseline epoch must cover every training row")\n            scheduler.step()\n            model.eval()\n            vcount, vcorrect, vloss = 0, 0, 0.\n            with torch.inference_mode():\n                for images, labels in val:\n                    images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)\n                    with torch.autocast(device_type=device.type, enabled=device.type == "cuda"):\n                        logits = model(images)\n                    vcount += len(labels)\n                    vcorrect += int(logits.argmax(1).eq(labels).sum())\n                    vloss += float(nn.functional.cross_entropy(logits.float(), labels, reduction="sum"))\n            history.append(dict(epoch=epoch, train_loss=loss_total/count, train_accuracy=correct/count,\n                                val_loss=vloss/vcount, val_accuracy=vcorrect/vcount))\n            base._json_write(output / f"{name}_history.json", history)\n            print(f"Baseline {name} {epoch}/{epochs}: val={vcorrect}/{vcount}", flush=True)\n        weights = output / f"{name}_fp32.pt"\n        torch.save(dict(state_dict={k: v.detach().cpu().clone() for k, v in model.state_dict().items()},\n                        classes=tables["classes"], seed=2026, size=64, epochs=epochs,\n                        data_signature=tables["data_signature"], trained_on="train"), weights)\n        row = dict(model=name, seed=2026, epochs=epochs, train_images=len(tables["train"]),\n                   validation_count=vcount, validation_correct=vcorrect, validation_accuracy=vcorrect/vcount,\n                   validation_loss=vloss/vcount, elapsed_minutes=(time.time()-started)/60,\n                   trained_on="train", weight_sha256=base._sha256(weights),\n                   outside_final_training_timer=True, fit_test_images=0)\n        rows.append(row)\n        base._json_write(output / "results.json", rows)\n        del model, optimizer, scheduler, scaler, train, val\n        gc.collect()\n        if device.type == "cuda":\n            torch.cuda.empty_cache()\n    return rows\n'}
EMBEDDED_HASHES = {'geo_worker.py': '11c96a5c281dfb3ec7b091d0a6d87672ff6eab5b65914d9471c4f69b34536463', 'geo_search.py': '7d97930208cf74f53429f32ec6a419494eb48ad0c804d58cf009090daaa661f8', 'geo_v7_45_worker.py': 'ecfcb8c97f7fad9a40b9cabbabee91ccf27b9b87c7eceacdf037e963c78638a9', 'geo_v7_45_search.py': '08e3242b9d413b6ea79740fc916cdf5d06d41d87f14ba4e11271f975c9a6edf9', 'geo_v7_45_baselines.py': '8ccc512a89f73a4a2bea02034a57b93387cef1133b9b9b999da2aca972f0c057'}

CODE_DIR.mkdir(parents=True, exist_ok=True)
for name, source in EMBEDDED_SOURCES.items():
    if hashlib.sha256(source.encode()).hexdigest() != EMBEDDED_HASHES[name]:
        raise RuntimeError("Изменился встроенный исходник: " + name)
    (CODE_DIR / name).write_text(source, encoding="utf-8")
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))
for name in EMBEDDED_SOURCES:
    sys.modules.pop(Path(name).stem, None)
importlib.invalidate_caches()
import geo_search
import geo_v7_45_worker as worker
import geo_v7_45_search as experiment
import geo_v7_45_baselines as baselines
from IPython.display import display, FileLink


In [ ]:
DATA_DIR = geo_search.prepare_data(DATA_DIR_OVERRIDE)
pool = experiment.Pool(DATA_DIR, RUN_DIR, CODE_DIR, CACHE_DIR)
worker.build_cache(DATA_DIR, CACHE_DIR, include_test=False)
print("Подготовлены только train и val; test не открыт.")


## Обязательное сравнение MLP и собственной CNN

Обе модели обучаются только на полном train, val используется только
для оценки. Seed установлен **до создания** каждой модели. Сохраняются
FP32-веса, loss/accuracy и длительность. Это предварительные эксперименты,
не финальное обучение, и эти модели не входят в итоговый ансамбль.


In [ ]:
BASELINE_RESULTS = baselines.run_baselines(
    DATA_DIR, RUN_DIR, CACHE_DIR, epochs=BASELINE_EPOCHS
)
for row in BASELINE_RESULTS:
    print(row)


## Предварительное обучение и выбор рецепта

ResNet152, ResNet101 и Inception v3: только train, до 14 фиксированных
эпох, последняя EMA, без выбора лучшей эпохи по val. Ранние ResNet-блоки
заморожены вместе с BN-статистиками. Аугментации стандартными torchvision
transforms, без собственной симуляции JPEG/DCT.

На полном val сравнивается ограниченный заранее заданный набор ансамблей
максимум из двух моделей и none/D4. Основная метрика — Accuracy, затем
log-loss. Рецепт фиксируется до доступа к test. Его оценка на уже использованном
для выбора val не является новым независимым тестом.


In [ ]:
SCREEN_FITS = experiment.run_screen(pool)
SELECTED_RECIPE = experiment.select_recipe(pool)
print("Зафиксированный рецепт:", SELECTED_RECIPE)
experiment.make_report(pool)


In [ ]:
import json, numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay
print("MLP/CNN — train-only оценка:")
display(pd.DataFrame(BASELINE_RESULTS))
print("Предобученные модели — train-only оценка:")
display(pd.DataFrame(experiment.read_json(RUN_DIR / "screen_model_metrics.json")))

fig, axes = plt.subplots(2, 2, figsize=(12, 7))
for row, name in enumerate(("mlp", "cnn")):
    hist = pd.DataFrame(experiment.read_json(RUN_DIR / "baselines" / f"{name}_history.json"))
    axes[row, 0].plot(hist.epoch, hist.train_loss, label="train")
    axes[row, 0].plot(hist.epoch, hist.val_loss, label="val")
    axes[row, 0].set_title(name + ": loss")
    axes[row, 1].plot(hist.epoch, hist.train_accuracy, label="train")
    axes[row, 1].plot(hist.epoch, hist.val_accuracy, label="val")
    axes[row, 1].set_title(name + ": accuracy")
for axis in axes.flat:
    axis.set_xlabel("epoch"); axis.legend(); axis.grid(alpha=.2)
fig.tight_layout()
fig.savefig(RUN_DIR / "baseline_learning_curves.png", dpi=120)
plt.show()

selected = SELECTED_RECIPE["selected"]
values = {}
for record in experiment.read_json(RUN_DIR / "screen_predictions.json"):
    job = experiment.read_json(Path(record["result_dir"]) / "job.json")
    if job["tta"] == selected["tta"] and record["key"] in selected["weights"]:
        values[record["key"]] = experiment.read_prediction(
            record, pool.val["id"].tolist(), pool.classes
        )["probs"]
guess = experiment.mixture(values, selected["weights"]).argmax(1)
truth = pool.val["label"].to_numpy()
cm = np.zeros((len(pool.classes), len(pool.classes)), dtype=np.int64)
np.add.at(cm, (truth, guess), 1)
pd.DataFrame(cm, index=pool.classes, columns=pool.classes).to_csv(RUN_DIR / "screen_confusion_matrix.csv")
fig, axis = plt.subplots(figsize=(12, 10))
ConfusionMatrixDisplay(cm, display_labels=pool.classes).plot(ax=axis, xticks_rotation=90, colorbar=False)
axis.set_title("Train-only screening ensemble: reused held-out val")
fig.tight_layout()
fig.savefig(RUN_DIR / "screen_confusion_matrix.png", dpi=120)
plt.show()
# Показываем только агрегаты. Ни test, ни изображения здесь не открываются.


## Финальное обучение и официальный таймер

Эта ячейка каждый раз запускает **новое** обучение выбранных моделей на
train + val. Старые финальные результаты не переиспользуются для таймера.
Максимум 14 эпох; при ограничении бюджета сохраняется только последняя
**полностью завершённая** эпоха, минимум 3. Реальное количество эпох
и причина остановки записываются. Это обучение по бюджету, а не обещание,
что любая остановленная модель прошла все 14 эпох.

Все выбранные модели обязательны. Если одна не достигла минимального
протокола или произошла ошибка, CSV не выдаётся за успешное решение.
У завершённых train+val моделей **нет независимой validation accuracy**.
Предварительную val accuracy им не приписываем.

Для точного восстановления CSV нужны сохранённые FP32-веса, рецепт и
совместимые версии библиотек: одних seeds недостаточно, особенно при
остановке по времени. Сохрани все результаты приватно внутри команды.


In [ ]:
import json
FINAL_TRAIN_ERROR = None
FINAL_FITS = None
start_time = time.time()
start_monotonic = time.monotonic()
try:
    FINAL_FITS = experiment.run_final_train(pool)
except (Exception, KeyboardInterrupt) as error:
    FINAL_TRAIN_ERROR = f"{type(error).__name__}: {error}"
    print("Финальное обучение не завершено:", FINAL_TRAIN_ERROR)
    traceback.print_exc()
finally:
    end_time = time.time()
    elapsed_minutes = (time.monotonic() - start_monotonic) / 60
    print(f"Elapsed time: {elapsed_minutes:.2f} minutes")
    FINAL_TIMER_VALID = elapsed_minutes <= 45 and FINAL_TRAIN_ERROR is None
    (RUN_DIR / "final_wallclock_notebook.json").write_text(json.dumps({
        "start_time": start_time, "end_time": end_time,
        "elapsed_minutes": elapsed_minutes, "limit_minutes": 45,
        "valid": FINAL_TIMER_VALID, "error": FINAL_TRAIN_ERROR,
        "includes_startup_training_save_cleanup": True,
        "includes_test_prediction": False,
    }, indent=2), encoding="utf-8")
if not FINAL_TIMER_VALID:
    print("Нет успешного финального решения в лимите; test-предсказание будет пропущено.")


## Единственное использование test: итоговый CSV

Предсказание — **после остановки таймера обучения**, только из зафиксированных
FP32-checkpoints. Здесь нет обновления модели, BN-статистик, калибровки,
поиска весов или балансировки классов test. Шаблон submission определяет
названия столбцов и порядок IDs. Если обучение не завершено, test не читается.


In [ ]:
ARCHIVE_STATUS = "not_attempted"
PREDICTION_VALID = False
try:
    if FINAL_TIMER_VALID:
        SUBMISSION = experiment.predict_final(pool)
        PREDICTION_VALID = True
        print("Итоговый CSV:", SUBMISSION)
    else:
        print("Test не открыт: отсутствует корректное финальное обучение.")
except (Exception, KeyboardInterrupt) as error:
    print("Ошибка предсказания:", type(error).__name__, str(error))
    traceback.print_exc()
finally:
    try:
        experiment.make_report(pool, archive_status="pending")
        ARCHIVE_STATUS = experiment.bundle(pool)
    except (Exception, KeyboardInterrupt) as error:
        ARCHIVE_STATUS = f"failed: {type(error).__name__}: {error}"
        print("Сохрани папку outputs с весами целиком, если recovery не создался.")
    experiment.make_report(pool, archive_status=ARCHIVE_STATUS)
print((RUN_DIR / experiment.REPORT_NAME).read_text(encoding="utf-8"))
for name in (experiment.SUBMISSION_NAME, experiment.REPORT_NAME, experiment.ARCHIVE_NAME):
    path = RUN_DIR / name
    if path.is_file() and (name != experiment.SUBMISSION_NAME or PREDICTION_VALID):
        display(FileLink(str(path)))


## Что сохранить

Папка **`/kaggle/working/geo_v7_45`**:

- `submission_geo_v7_45.csv` — основной CSV, только после успешного финального обучения.
- `REPORT_V7_45_TO_SEND.txt` — агрегированный отчёт, его можно прислать в чат.
- `geo_v7_45_recovery.tar.gz` — веса/код/рецепты, сохранить приватно.
- `final_wallclock_notebook.json` — замер финального обучения.

Исходные изображения и uint8-кэш не включаются в recovery; не отправляй
в ИИ-чат изображения, исходные таблицы и индивидуальные прогнозы.
Для Moodle приложи notebook, все шаги решения, сравнение MLP/CNN/pretrained,
фактические эпохи/версии/seeds и точное обозначение выбранного submission.
Не публикуй приватные материалы и не передавай их другим командам.

Этот код ограничивает время вычислений, но не обещает заранее public/private
score или то, что Kaggle запустит сессию без очереди и внешних сбоев.
